# Mission 3 ver9 — 한국어 DeBERTa-v3 + NB-log-count LR

**ver8에서 쓰던 `mission3_ver5_inputs.zip`과 이 노트북 한 개로 실행합니다.**

1. 공유 ZIP에서 **정리 → 바로가기 추가 → 내 드라이브**를 선택합니다.
2. 내 드라이브 바로 아래에 `mission3_ver5_inputs.zip` 바로가기를 둡니다.
3. 이 노트북을 Colab에서 열고 **GPU 런타임 → 모두 실행**을 선택합니다.
4. 바로가기를 만든 계정으로 Drive를 연결합니다.

결과: `내 드라이브/DCC_AIdeas/results/mission3/ver9`.
새 모델: [한국어 DeBERTa-v3-base](https://huggingface.co/team-lucid/deberta-v3-base-korean),
[원논문](https://arxiv.org/abs/2111.09543).
문자·단어 TF-IDF를 [NB 로그비율](https://aclanthology.org/P12-2018/)로 재가중한 Logistic Regression도 추가합니다.

- 기존 ver4의 5개 모델은 비교·최종 혼합용으로 재사용합니다.
- 새로운 encoder 전체를 최대 3 epoch 학습합니다. 시간 여유가 충분하면 두 번째 fold도 학습합니다.
- 새 후보는 `0.75 × DeBERTa 평균 + 0.25 × NB-LR`로 고정합니다.
- Training 내부 교차평가에서 `+0.002 F1`, 최소 3개 fold 개선을 모두 만족할 때 후보 혼합을 채택합니다.
- 공식 Validation은 선택을 끝낸 뒤 평가에만 사용합니다. 실패하면 원래 ver4 보정값을 그대로 유지합니다.

**설정 셀부터 모든 자동 단계가 하나의 3시간 예산을 공유합니다.** 설치·복원·다운로드도 포함하며,
최종 추론에 25분을 남깁니다. GPU·네트워크·Drive 속도에 따라 학습 fold/epoch 수가 줄거나 평가가 미완료될 수 있습니다.
시간 초과 시 자식 프로세스를 종료하고 완료된 파일을 보존합니다. 연결 대기·수동 셀 간 대기 역시 시작 후 예산에서 차감됩니다.
최종 CSV를 먼저 저장하고 남은 시간에 후보 단독 진단을 수행합니다.
기본 결과 묶음은 기존 ver4를 포함하므로 원래 결과와 별도로 Drive **약 10~15 GB 이상** 공간을 확보하세요.

**실제 ver9 F1과 GPU 소요 시간은 실행 전에는 알 수 없습니다.** 마지막 셀에서 측정 결과와 선택 사유를 확인하세요.


In [ ]:
import time
RUN_STARTED = time.time()  # 이후 셀마다 초기화하지 않는 공통 시작 시각
INPUT_ZIP = "/content/drive/MyDrive/mission3_ver5_inputs.zip"
OUTPUT_DIR = "/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver9"
RUN_VALIDATION = True
RUN_CANDIDATE_DIAGNOSTIC = True
TRAIN_CONFIG = {
    "max_models": 2, "max_epochs": 3, "patience": 2,
    "learning_rate": 2e-5, "head_learning_rate": 1e-4,
    "final_reserve_seconds": 1500, "calibration_reserve_seconds": 600,
    "min_calibration_gain": 0.002, "min_calibration_improved_folds": 3,
    "keep_completed_resume_states": False,
}
INFERENCE_BATCH_SIZE = 4
# 다른 실험 설정으로 바꾸면 OUTPUT_DIR도 새 폴더로 바꾸세요.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False, timeout_ms=120000)
print(f"설정 이후 {(time.time() - RUN_STARTED) / 60:.1f}분 경과")


In [ ]:
# @title 노트북에 포함된 실행 코드 복원 (별도 소스 업로드 불필요)
import importlib.util
import json
import sys
import tempfile
from pathlib import Path
RUNTIME_SOURCES = {'__init__.py': '"""Mission 3 ver9: Korean DeBERTa and NB features under a wall-clock budget."""\n', 'assets.py': '"""Verify and package the original Training-only ver4 baseline."""\nfrom __future__ import annotations\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nif __package__:\n    from .ver4_runtime import pipeline as base_pipeline\n    from .ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, text_group\nelse:\n    from ver4_runtime import pipeline as base_pipeline\n    from ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, text_group\n\ndef _sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\ndef _release():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\ndef _versions():\n    return {name: importlib.metadata.version(name) for name in\n            ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}\n\ndef _check_versions(base_manifest):\n    current = _versions()\n    # sklearn objects in sparse.joblib require the original compatible environment.\n    for name in ("transformers", "numpy", "scikit-learn"):\n        saved = base_manifest.get("versions", {}).get(name)\n        if saved and current[name] != saved:\n            raise RuntimeError(f"{name}: ver4 저장 버전={saved}, 현재={current[name]}. requirements.txt를 설치하세요.")\n    return current\n\ndef _read_base(train_label_dir, base_dir, init_fold):\n    """Verify the exact original training run before trusting cached predictions."""\n    checkpoint = base_dir / "mission3_ver4_model.ckpt"\n    manifest = base_pipeline.load_manifest(checkpoint)\n    _check_versions(manifest)\n    print("기존 ver4 확인 완료. 서울 Training JSON을 읽습니다...", flush=True)\n    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)\n    print(f"Training {len(paths):,}개 로드 완료. 원래 데이터·분할·보정 캐시를 검증합니다...", flush=True)\n    fingerprint = base_pipeline._fingerprint(paths, texts, targets, manifest["config"])\n    if fingerprint != manifest.get("signature"):\n        raise ValueError("ver4와 Training 데이터/전처리/코드가 다릅니다. 원래 Training과 ver4 결과 묶음을 사용하세요.")\n    splits = json.loads((base_dir / "splits.json").read_text(encoding="utf-8"))\n    lookup = {p.name: i for i, p in enumerate(paths)}\n    all_names = set(lookup)\n    group_ids = np.asarray([text_group(text) for text in texts])\n\n    def indices(names, purpose):\n        if not names or len(names) != len(set(names)) or not set(names) <= all_names:\n            raise ValueError(f"Invalid saved {purpose} filenames")\n        return np.asarray([lookup[name] for name in names], dtype=np.int64)\n\n    cal = indices(splits["calibration"], "calibration")\n    if len(splits["folds"]) != len(manifest["folds"]) or not 1 <= init_fold <= len(splits["folds"]):\n        raise ValueError("init_fold / saved folds mismatch")\n    fold_indices = []\n    for number, fold in enumerate(splits["folds"], 1):\n        fit, stop = indices(fold["fit"], "fit"), indices(fold["stop"], "stop")\n        sets = [set(fit), set(stop), set(cal)]\n        groups = [set(group_ids[x]) for x in (fit, stop, cal)]\n        if any(sets[i] & sets[j] or groups[i] & groups[j] for i, j in ((0, 1), (0, 2), (1, 2))):\n            raise ValueError(f"fold {number}: sample or text-group leakage")\n        if set.union(*sets) != set(range(len(paths))):\n            raise ValueError(f"fold {number}: saved partitions do not cover Training")\n        fold_indices.append((fit, stop))\n    # Verify the stored fold identities against the ORIGINAL deterministic run.\n    # Preserve the old models\' exclusions and the comparable single-fold stop.\n    _, expected_cal, expected_folds, _ = base_pipeline.make_splits(\n        texts, targets, manifest["config"]["calibration_size"],\n        manifest["config"]["n_folds"], manifest["config"]["seed"],\n    )\n    if not np.array_equal(cal, expected_cal) or any(\n        not np.array_equal(actual, expected)\n        for pair, reference in zip(fold_indices, expected_folds)\n        for actual, expected in zip(pair, reference)\n    ):\n        raise ValueError("Saved fold identity differs from the original ver4 run; base reuse would leak.")\n    for number, name in enumerate(manifest["folds"], 1):\n        saved = torch.load(base_dir / name, map_location="cpu", weights_only=True, mmap=True)\n        if (saved.get("format_version") != 1\n                or saved.get("seed") != manifest["config"]["seed"] + number * 1009\n                or saved.get("architecture", {}).get("num_labels") != len(TARGET_SYMPTOMS)):\n            raise ValueError(f"Saved checkpoint fold identity mismatch: {name}")\n        del saved\n    with np.load(base_dir / "calibration_predictions.npz", allow_pickle=False) as cache:\n        names = cache["filenames"].tolist()\n        if len(set(names)) != len(names) or set(names) != set(splits["calibration"]):\n            raise ValueError("ver4 calibration cache filenames mismatch")\n        order = {name: i for i, name in enumerate(names)}\n        aligned = [order[paths[i].name] for i in cal]\n        raw_neural = np.array(cache["neural"][aligned], dtype=np.float64)\n        raw_sparse = np.array(cache["sparse"][aligned], dtype=np.float64)\n        if not np.array_equal(cache["targets"][aligned], targets[cal]):\n            raise ValueError("ver4 calibration cache targets mismatch")\n    for values in (raw_neural, raw_sparse):\n        if values.shape != (len(cal), len(TARGET_SYMPTOMS)) or not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):\n            raise ValueError("Invalid ver4 raw calibration probabilities")\n    fit, stop = fold_indices[init_fold - 1]\n    return manifest, paths, texts, targets, fit, stop, cal, group_ids, raw_neural, raw_sparse\n\ndef _copy_file(source, target, expected_hash):\n    """An interrupted copy never appears under its final filename."""\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        if target.stat().st_size != source.stat().st_size or _sha256(target) != expected_hash:\n            raise ValueError(f"기존 묶음 파일 내용이 다릅니다: {target}")\n        return\n    temporary = target.with_name(target.name + ".tmp")\n    digest = hashlib.sha256()\n    with source.open("rb") as reader, temporary.open("wb") as writer:\n        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):\n            writer.write(chunk)\n            digest.update(chunk)\n    if digest.hexdigest() != expected_hash:\n        temporary.unlink()\n        raise ValueError(f"복사 중 기본 모델 파일이 변경되었습니다: {source}")\n    temporary.replace(target)\n\ndef _export_base(base_dir, output_dir, manifest, asset_hashes):\n    destination = output_dir / "base"\n    destination.mkdir(exist_ok=True)\n    # Copy to make the resulting ver9 folder portable without its old sibling.\n    for name in [*manifest["folds"], manifest["sparse"], "mission3_ver4_model.ckpt"]:\n        print("Bundle:", name, flush=True)\n        _copy_file(base_dir / name, destination / name, asset_hashes[name])\n    tokenizer_dir = base_dir / manifest["tokenizer"]\n    for path in tokenizer_dir.rglob("*"):\n        if path.is_file():\n            _copy_file(path, destination / manifest["tokenizer"] / path.relative_to(tokenizer_dir),\n                       asset_hashes[str(path.relative_to(base_dir))])\n    copied = base_pipeline.load_manifest(destination / "mission3_ver4_model.ckpt")\n    if copied["signature"] != manifest["signature"]:\n        raise ValueError("Portable base bundle signature mismatch")\n\ndef _checked_probabilities(values, rows):\n    values = np.asarray(values, dtype=np.float64)\n    if (values.shape != (rows, len(TARGET_SYMPTOMS)) or not np.isfinite(values).all()\n            or np.any((values < 0) | (values > 1))):\n        raise ValueError("Invalid cached/model probabilities")\n    return values\n\ndef _cached_predictions(path, signature, filenames, weights_digest, compute):\n    """Bind a Training diagnostic cache to its exact model and held-out rows."""\n    if path.is_file():\n        with np.load(path, allow_pickle=False) as cache:\n            if (cache["signature"].item() != signature or cache["filenames"].tolist() != filenames\n                    or cache["weights_sha256"].item() != weights_digest):\n                raise ValueError("Training prediction cache mismatch")\n            values = np.array(cache["probabilities"])\n    else:\n        values = _checked_probabilities(compute(), len(filenames))\n        temporary = path.with_name(path.name + ".tmp")\n        with temporary.open("wb") as stream:\n            np.savez_compressed(stream, probabilities=values, filenames=np.asarray(filenames),\n                                signature=np.asarray(signature), weights_sha256=np.asarray(weights_digest))\n        temporary.replace(path)\n    return _checked_probabilities(values, len(filenames))\n\ndef _tokenizer_hashes(directory):\n    directory = Path(directory)\n    return {p.relative_to(directory).as_posix(): _sha256(p)\n            for p in sorted(directory.rglob("*")) if p.is_file()}\n', 'budget.py': '"""One shared wall-clock budget for setup, training and evaluation.\n\nCreate ``budget = TimeBudget()`` before the first automatic notebook step; pass\n``start_epoch`` and ``deadline_epoch`` to every child and reconstruct with\n``TimeBudget.from_deadline(start_epoch, deadline_epoch)``. Never create a fresh\nthree-hour budget for a later phase. ``check(reserve_seconds=1500)`` returns the\nusable seconds before the reserved final phase, or raises ``BudgetExceeded``.\n``run`` inherits stdout/stderr by default and returns a CompletedProcess. It\nstops the entire child process group on timeout (Colab/Linux), retaining files\nalready written by the caller. Normal nonzero exits follow subprocess.run\'s\n``check`` convention. No machine-learning dependencies are imported here.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport codecs\nimport os\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom typing import Callable, Sequence\n\n\nMAX_SECONDS = 3 * 60 * 60\nSAFETY_SECONDS = 5.0\nTERMINATION_SECONDS = 2.0\n\n\nclass BudgetExceeded(TimeoutError):\n    """A phase cannot continue within its shared absolute deadline."""\n\n\nclass _OutputRelay:\n    """Forward a pipe without tying timeout handling to a blocking readline.\n\n    Read the descriptor directly in nonblocking mode. Closing a TextIOWrapper\n    while another thread holds its read lock can hang when a descendant keeps\n    the pipe open, so this relay never calls the wrapper\'s blocking read APIs.\n    """\n\n    def __init__(self, stream):\n        self.stream = stream\n        self.output = sys.stdout\n        self.cancel = threading.Event()\n        self.finished = threading.Event()\n        self.error = None\n        self.thread = threading.Thread(target=self._forward, name="mission3-output", daemon=True)\n        os.set_blocking(stream.fileno(), False)\n\n    def start(self):\n        self.thread.start()\n\n    def _forward(self):\n        decoder = codecs.getincrementaldecoder(self.stream.encoding or "utf-8")(\n            errors=self.stream.errors or "replace")\n        try:\n            while not self.cancel.is_set():\n                try:\n                    chunk = os.read(self.stream.fileno(), 65536)\n                except BlockingIOError:\n                    self.cancel.wait(0.02)\n                    continue\n                if not chunk:\n                    tail = decoder.decode(b"", final=True)\n                    if tail:\n                        self.output.write(tail)\n                        self.output.flush()\n                    break\n                text = decoder.decode(chunk, final=False)\n                if text:\n                    self.output.write(text)\n                    self.output.flush()\n        except BaseException as error:\n            if not self.cancel.is_set():\n                self.error = error\n        finally:\n            self.finished.set()\n\n    def close(self):\n        self.cancel.set()\n        # The nonblocking reader exits promptly even if an escaped descendant\n        # retains stdout. A stalled notebook output writer is a daemon thread.\n        self.thread.join(timeout=0.1)\n\n\ndef _finite_seconds(value: float, name: str, *, positive: bool = False) -> float:\n    try:\n        value = float(value)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(f"{name} must be a finite number") from exc\n    if not math.isfinite(value) or (value <= 0 if positive else value < 0):\n        condition = "positive" if positive else "nonnegative"\n        raise ValueError(f"{name} must be finite and {condition}")\n    return value\n\n\nclass TimeBudget:\n    """Absolute epoch deadline, capped at 10,800 seconds from the run start.\n\n    ``remaining_seconds`` includes the five-second shutdown safety margin;\n    ``check`` and ``run`` exclude it. ``reserve_seconds`` additionally leaves\n    time for subsequent phases. ``clock`` is injectable for deterministic tests.\n    A monotonic bound prevents a backward wall-clock adjustment extending a run.\n    """\n\n    def __init__(\n        self,\n        *,\n        start_epoch: float | None = None,\n        total_seconds: float = MAX_SECONDS,\n        deadline_epoch: float | None = None,\n        clock: Callable[[], float] = time.time,\n        monotonic_clock: Callable[[], float] = time.monotonic,\n    ) -> None:\n        total_seconds = _finite_seconds(total_seconds, "total_seconds", positive=True)\n        if total_seconds > MAX_SECONDS:\n            raise ValueError(f"total_seconds cannot exceed {MAX_SECONDS}")\n        now = clock()\n        self.start_epoch = _finite_seconds(now if start_epoch is None else start_epoch, "start_epoch")\n        self.deadline_epoch = _finite_seconds(\n            self.start_epoch + total_seconds if deadline_epoch is None else deadline_epoch,\n            "deadline_epoch",\n        )\n        if not self.start_epoch < self.deadline_epoch <= self.start_epoch + total_seconds:\n            raise ValueError("deadline_epoch must be after start_epoch and within total_seconds")\n        if self.start_epoch > now:\n            raise ValueError("start_epoch cannot be in the future")\n        self._clock = clock\n        self._monotonic_clock = monotonic_clock\n        self._monotonic_deadline = monotonic_clock() + max(0.0, self.deadline_epoch - now)\n\n    @classmethod\n    def from_deadline(cls, start_epoch: float, deadline_epoch: float, **kwargs) -> "TimeBudget":\n        """Reconstruct the same deadline in a child; does not restart the clock."""\n        return cls(start_epoch=start_epoch, deadline_epoch=deadline_epoch, **kwargs)\n\n    def remaining_seconds(self) -> float:\n        """Seconds remaining to the absolute deadline, clamped to zero."""\n        return max(0.0, min(\n            self.deadline_epoch - self._clock(),\n            self._monotonic_deadline - self._monotonic_clock(),\n        ))\n\n    def check(self, reserve_seconds: float = 0, *, phase: str = "work") -> float:\n        """Return usable seconds, excluding shutdown margin and phase reserve."""\n        reserve_seconds = _finite_seconds(reserve_seconds, "reserve_seconds")\n        usable = self.remaining_seconds() - SAFETY_SECONDS - reserve_seconds\n        if usable <= 0:\n            raise BudgetExceeded(f"No time left for {phase}; preserving existing artifacts")\n        return usable\n\n    def run(\n        self,\n        command: Sequence[str],\n        *,\n        reserve_seconds: float = 0,\n        timeout: float | None = None,\n        check: bool = True,\n        capture_output: bool = False,\n        stream_output: bool = False,\n        **popen_kwargs,\n    ) -> subprocess.CompletedProcess:\n        """Run a command within this budget, killing its process group if needed.\n\n        ``timeout`` can shorten a phase but cannot extend the shared deadline.\n        Standard Popen options such as ``cwd``, ``env``, ``text``, ``stdout`` and\n        ``stderr`` are accepted. ``start_new_session`` is always true so timeout\n        cleanup cannot signal the notebook\'s own process group.\n\n        ``stream_output=True`` forwards merged stdout/stderr to Python\'s current\n        stdout, which makes child logs visible in an in-process Colab notebook.\n        It does not retain an unbounded copy of training output. A separate\n        thread reads the pipe while the supervisor independently enforces the\n        process deadline. It cannot be combined with ``capture_output``.\n        """\n        usable = self.check(reserve_seconds, phase="subprocess")\n        if timeout is not None:\n            timeout = _finite_seconds(timeout, "timeout", positive=True)\n            usable = min(usable, timeout)\n        if "start_new_session" in popen_kwargs:\n            raise ValueError("start_new_session is managed by TimeBudget.run")\n        if stream_output and capture_output:\n            raise ValueError("stream_output cannot be combined with capture_output")\n        if stream_output:\n            if "stdout" in popen_kwargs or "stderr" in popen_kwargs:\n                raise ValueError("stream_output cannot be combined with stdout or stderr")\n            if popen_kwargs.get("text") is False or popen_kwargs.get("universal_newlines") is False:\n                raise ValueError("stream_output requires text mode")\n            popen_kwargs.update(stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n            popen_kwargs.setdefault("errors", "replace")\n        if capture_output:\n            if "stdout" in popen_kwargs or "stderr" in popen_kwargs:\n                raise ValueError("capture_output cannot be combined with stdout or stderr")\n            popen_kwargs.update(stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n        phase_deadline = time.monotonic() + usable\n        process = subprocess.Popen(command, start_new_session=True, **popen_kwargs)\n        relay = None\n        try:\n            if stream_output:\n                relay = _OutputRelay(process.stdout)\n                relay.start()\n            usable = min(\n                self.check(reserve_seconds, phase="subprocess"),\n                phase_deadline - time.monotonic(),\n            )\n            if usable <= 0:\n                raise BudgetExceeded("Subprocess startup exhausted the available budget")\n            if relay is None:\n                stdout, stderr = process.communicate(timeout=usable)\n            else:\n                # Never block the supervisor on output, line endings or EOF.\n                process.wait(timeout=usable)\n                drain_seconds = min(self.check(reserve_seconds, phase="output drain"),\n                                    phase_deadline - time.monotonic())\n                if not relay.finished.wait(timeout=max(0.0, drain_seconds)):\n                    raise subprocess.TimeoutExpired(command, usable)\n                if relay.error is not None:\n                    raise relay.error\n                stdout, stderr = None, None\n        except subprocess.TimeoutExpired as exc:\n            self._stop(process)\n            failure = BudgetExceeded(f"Subprocess exceeded its time budget: {command!r}")\n            failure.stdout, failure.stderr = exc.stdout, exc.stderr\n            raise failure from exc\n        except BaseException:\n            self._stop(process)\n            raise\n        finally:\n            if relay is not None:\n                relay.close()\n            for stream in (process.stdout, process.stderr):\n                if stream is not None:\n                    stream.close()\n        result = subprocess.CompletedProcess(command, process.returncode, stdout, stderr)\n        if check:\n            result.check_returncode()\n        return result\n\n    def _stop(self, process: subprocess.Popen) -> None:\n        """TERM then KILL the session\'s process group within two seconds."""\n        stop_by = time.monotonic() + min(TERMINATION_SECONDS, self.remaining_seconds())\n\n        def send(sig: int) -> None:\n            try:\n                if os.name == "posix":\n                    # Signal even if the group leader exited: grandchildren may remain.\n                    os.killpg(process.pid, sig)\n                elif process.poll() is None:\n                    process.terminate() if sig == signal.SIGTERM else process.kill()\n            except ProcessLookupError:\n                pass\n\n        send(signal.SIGTERM)\n        try:\n            process.wait(timeout=max(0.0, stop_by - time.monotonic() - 0.25))\n        except subprocess.TimeoutExpired:\n            pass\n        finally:\n            # A TERM-ignoring child can survive even after its parent has exited.\n            send(getattr(signal, "SIGKILL", signal.SIGTERM))\n        try:\n            process.wait(timeout=max(0.0, stop_by - time.monotonic()))\n        except subprocess.TimeoutExpired:\n            # KILL was already sent; never let OS-level reaping extend the deadline.\n            pass\n\n\n# Both names describe the same shared, absolute budget.\nAbsoluteDeadline = TimeBudget\n', 'bootstrap.py': '"""Restore the one-file Mission 3 input bundle using only the standard library.\n\nThe uploaded ZIP remains in Drive. Extraction uses a fixed local directory so\nepoch resume sees the same initial checkpoint path across Colab sessions.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\n\nBUNDLE_NAME = "mission3_ver5_inputs.zip"\nMODEL_PREFIX = "DCC_AIdeas/results/mission3/ver4/"\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not name or "\\\\" in name or ":" in name or path.is_absolute()\n            or ".." in path.parts or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    path = root / _relative(name)\n    if not path.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    return path\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for data in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(data)\n    return digest.hexdigest()\n\n\ndef _files(manifest, archive):\n    if manifest.get("format") != "mission3_ver5_inputs_v1":\n        raise ValueError("Unsupported Mission 3 input bundle format")\n    files = manifest.get("files")\n    if not isinstance(files, list) or not files:\n        raise ValueError("Bundle manifest has no files")\n    names = set()\n    for row in files:\n        name = row["path"]\n        _relative(name)\n        if name in names or not name.startswith("DCC_AIdeas/"):\n            raise ValueError(f"Duplicate or unexpected bundle member: {name}")\n        names.add(name)\n        if (type(row["size"]) is not int or row["size"] < 0\n                or not re.fullmatch(r"[0-9a-f]{64}", row["sha256"])):\n            raise ValueError(f"Invalid manifest metadata: {name}")\n        info = archive.getinfo(name)\n        if info.is_dir() or info.file_size != row["size"]:\n            raise ValueError(f"Bundle size mismatch: {name}")\n    required = {MODEL_PREFIX + name for name in\n                ["mission3_ver4_model.ckpt", "splits.json", "calibration_predictions.npz", "sparse.joblib",\n                 *[f"fold_{i}.ckpt" for i in range(1, 6)], "tokenizer/tokenizer.json",\n                 "tokenizer/tokenizer_config.json", "tokenizer/vocab.txt", "tokenizer/special_tokens_map.json"]}\n    if not required <= names:\n        raise ValueError(f"Missing ver4 inputs: {sorted(required - names)}")\n    for prefix, key in [(TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")]:\n        count = sum(name.startswith(prefix) and name.endswith(".json") for name in names)\n        if count < 1 or type(manifest.get(key)) is not int or count != manifest[key]:\n            raise ValueError(f"Bundle JSON count mismatch: {key}")\n    infos = archive.infolist()\n    if len({info.filename for info in infos}) != len(infos):\n        raise ValueError("Duplicate ZIP member")\n    allowed_files = names | {"bundle_manifest.json", "README_UPLOAD.txt"}\n    for info in infos:\n        _relative(info.filename)\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n        if not info.is_dir() and info.filename not in allowed_files:\n            raise ValueError(f"Unlisted bundle member: {info.filename}")\n    return files\n\n\ndef restore_input_bundle(archive_path, destination):\n    """Verify SHA256 while extracting; reuse unchanged files from a verified run."""\n    archive_path = Path(archive_path).expanduser().resolve()\n    root = Path(destination).expanduser().resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / ".mission3_bundle_complete.json"\n    with zipfile.ZipFile(archive_path) as archive:\n        manifest_bytes = archive.read("bundle_manifest.json")\n        manifest = json.loads(manifest_bytes)\n        files = _files(manifest, archive)\n        signature = hashlib.sha256(manifest_bytes).hexdigest()\n        previous = {}\n        try:\n            saved = json.loads(marker.read_text(encoding="utf-8"))\n            if saved.get("manifest_sha256") == signature:\n                previous = saved["files"]\n        except (OSError, ValueError, KeyError):\n            pass\n        pending, state = [], {}\n        for row in files:\n            target = _target(root, row["path"])\n            current = target.stat() if target.is_file() else None\n            stamp = [current.st_size, current.st_mtime_ns] if current else None\n            if stamp and current.st_size == row["size"] and previous.get(row["path"]) == stamp:\n                state[row["path"]] = stamp\n            else:\n                pending.append(row)\n        if not pending:\n            print("입력 ZIP: 이전에 검증한 Colab 로컬 파일을 재사용합니다.", flush=True)\n            return root / "DCC_AIdeas"\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Colab 로컬 저장소에 입력 ZIP을 풀 공간이 부족합니다.")\n        print(f"입력 ZIP 확인·압축 해제: {len(pending):,}개 파일 (WAV 없음)", flush=True)\n        for index, row in enumerate(pending, 1):\n            target = _target(root, row["path"])\n            target.parent.mkdir(parents=True, exist_ok=True)\n            # Recover completed files from an interrupted extraction without copying them again.\n            verified = target.is_file() and target.stat().st_size == row["size"] and _hash(target) == row["sha256"]\n            if not verified:\n                temporary = None\n                try:\n                    digest, size = hashlib.sha256(), 0\n                    with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                            dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                        temporary = Path(writer.name)\n                        for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):\n                            writer.write(chunk)\n                            digest.update(chunk)\n                            size += len(chunk)\n                    if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                        raise ValueError(f"ZIP 내용 검증 실패: {row[\'path\']}")\n                    temporary.replace(target)\n                finally:\n                    if temporary is not None:\n                        temporary.unlink(missing_ok=True)\n            current = target.stat()\n            state[row["path"]] = [current.st_size, current.st_mtime_ns]\n            if row["size"] > 100 * 1024 * 1024 or index % 2000 == 0 or index == len(pending):\n                print(f"압축 해제 {index:,}/{len(pending):,}: {target.name}", flush=True)\n        # A marker is published only after every file has passed hash verification.\n        temporary = marker.with_suffix(".tmp")\n        temporary.write_text(json.dumps({"manifest_sha256": signature, "files": state}), encoding="utf-8")\n        temporary.replace(marker)\n    print(f"준비 완료: Training {manifest[\'training_count\']:,}개 / Validation {manifest[\'validation_count\']:,}개", flush=True)\n    return root / "DCC_AIdeas"\n', 'calibration.py': '"""Training-only grouped selection of ver4 versus a fixed ver9 candidate.\n\nThe candidate probabilities are supplied by the caller: the prespecified\nDeBERTa/NB-logistic mixture is not tuned again inside this module.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nif __package__:\n    from .ver4_runtime.calibration import (\n        WEIGHT_GRID, _best_threshold, _grouped_splits, _inputs, _probabilities,\n        apply_calibrator as _apply_baseline, fit_calibrator as _fit_baseline,\n    )\n    from .ver4_runtime.data import TARGET_SYMPTOMS\nelse:\n    from ver4_runtime.calibration import (\n        WEIGHT_GRID, _best_threshold, _grouped_splits, _inputs, _probabilities,\n        apply_calibrator as _apply_baseline, fit_calibrator as _fit_baseline,\n    )\n    from ver4_runtime.data import TARGET_SYMPTOMS\n\nCANDIDATE_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)\nLOGIT_EPSILON = 1e-6\n\n\ndef _blend_space(value):\n    if value not in ("raw", "logit"):\n        raise ValueError("blend_space must be \'raw\' or \'logit\'")\n    return value\n\n\ndef _logits(probabilities):\n    probabilities = np.clip(np.asarray(probabilities, dtype=np.float64),\n                            LOGIT_EPSILON, 1.0 - LOGIT_EPSILON)\n    return np.log(probabilities) - np.log1p(-probabilities)\n\n\ndef _sigmoid(logits):\n    # Convex mixtures of clipped logits stay within finite, safe exp bounds.\n    return 1.0 / (1.0 + np.exp(-logits))\n\n\ndef _validated_inputs(y_true, base_neural, sparse, candidate):\n    truth, base, sparse = _inputs(y_true, base_neural, sparse)\n    candidate = _probabilities(candidate, "candidate")\n    if candidate.shape != base.shape:\n        raise ValueError("candidate and base_neural must have identical shapes")\n    return truth, base, sparse, candidate\n\n\ndef _with_zero_candidate(calibrator):\n    return {**calibrator, "candidate_weights": [0.0] * len(TARGET_SYMPTOMS),\n            "blend_space": "raw"}\n\n\ndef _fit_mixture(y_true, base_neural, sparse, candidate, blend_space="raw"):\n    """Exact F1 selection per label; equal scores prefer zero candidate weight."""\n    blend_space = _blend_space(blend_space)\n    truth, base, sparse, candidate = _validated_inputs(y_true, base_neural, sparse, candidate)\n    if blend_space == "logit":\n        base, sparse, candidate = map(_logits, (base, sparse, candidate))\n    alphas, betas, thresholds = [], [], []\n    for column in range(len(TARGET_SYMPTOMS)):\n        best = None\n        for alpha in WEIGHT_GRID:\n            baseline = alpha * base[:, column] + (1.0 - alpha) * sparse[:, column]\n            for beta in CANDIDATE_GRID:\n                probabilities = (1.0 - beta) * baseline + beta * candidate[:, column]\n                if blend_space == "logit":\n                    probabilities = _sigmoid(probabilities)\n                threshold, score = _best_threshold(truth[:, column], probabilities)\n                # Keep ver4\'s alpha tie convention after preferring smaller beta.\n                choice = (score, -beta, -abs(alpha - 0.5), alpha, threshold)\n                if best is None or choice > best:\n                    best = choice\n        _, negative_beta, _, alpha, threshold = best\n        alphas.append(float(alpha))\n        betas.append(float(-negative_beta))\n        thresholds.append(float(threshold))\n    return {"neural_weights": alphas, "candidate_weights": betas,\n            "thresholds": thresholds, "blend_space": blend_space}\n\n\ndef apply_selected(base_neural, base_sparse, candidate, selection):\n    """Apply a stored selection; zero candidate weight preserves raw ver4 exactly."""\n    sparse, calibrator = base_sparse, selection\n    blend_space = _blend_space(calibrator.get("blend_space", "raw"))\n    # Reuse ver4\'s shape, range, alpha and threshold validation in both spaces.\n    baseline = _apply_baseline(base_neural, sparse, calibrator)\n    betas = np.asarray(calibrator["candidate_weights"], dtype=np.float64)\n    if betas.shape != (len(TARGET_SYMPTOMS),) or not np.isfinite(betas).all():\n        raise ValueError("candidate_weights must contain one finite weight per symptom")\n    if np.any((betas < 0) | (betas > 1)):\n        raise ValueError("candidate_weights must be in [0, 1]")\n    if blend_space == "logit":\n        alphas = np.asarray(calibrator["neural_weights"], dtype=np.float64)\n        baseline = alphas * _logits(base_neural) + (1.0 - alphas) * _logits(sparse)\n    if candidate is None:\n        if np.any(betas != 0):\n            raise ValueError("candidate probabilities are required for nonzero candidate_weights")\n        return _sigmoid(baseline) if blend_space == "logit" else baseline\n    candidate = _probabilities(candidate, "candidate")\n    if candidate.shape != baseline.shape:\n        raise ValueError("candidate and base_neural must have identical shapes")\n    if blend_space == "logit":\n        candidate = _logits(candidate)\n    mixture = (1.0 - betas) * baseline + betas * candidate\n    return _sigmoid(mixture) if blend_space == "logit" else mixture\n\n\ndef _metrics(truth, predictions):\n    precision, recall, f1, support = precision_recall_fscore_support(\n        truth, predictions, average=None, zero_division=0,\n    )\n    return {\n        symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}\n        for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)\n    }\n\n\ndef fit_candidate_only_crossfit(y_true, candidate, groups, seed=42, n_splits=5):\n    """Diagnose a standalone candidate using grouped Training calibration only.\n\n    The candidate model must exclude all supplied samples from model fitting and\n    epoch selection. Each scored fold uses thresholds fitted on the other\n    groups; the returned thresholds are then fitted on all Training calibration\n    rows. Never supply official Validation. This diagnostic does not participate\n    in the baseline/mixture selection procedure or its adoption guard.\n    """\n    candidate = _probabilities(candidate, "candidate")\n    truth, candidate, _ = _inputs(y_true, candidate, candidate)\n\n    def fit_thresholds(rows):\n        return [\n            float(_best_threshold(truth[rows, column], candidate[rows, column])[0])\n            for column in range(len(TARGET_SYMPTOMS))\n        ]\n\n    predictions = np.zeros_like(truth)\n    seen = np.zeros(len(truth), dtype=np.int64)\n    fold_count = 0\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        thresholds = np.asarray(fit_thresholds(train), dtype=np.float64)\n        predictions[test] = candidate[test] >= thresholds\n        np.add.at(seen, test, 1)\n        fold_count += 1\n    if not np.all(seen == 1):\n        raise RuntimeError("every calibration sample must be scored exactly once")\n\n    calibrator = {"thresholds": fit_thresholds(np.arange(len(truth)))}\n    fitted = candidate >= np.asarray(calibrator["thresholds"], dtype=np.float64)\n    report = {\n        "candidate_standalone_crossfit_macro_f1": float(\n            f1_score(truth, predictions, average="macro", zero_division=0)),\n        "candidate_standalone_fit_macro_f1": float(\n            f1_score(truth, fitted, average="macro", zero_division=0)),\n        "candidate_standalone_per_symptom_crossfit": _metrics(truth, predictions),\n        "candidate_standalone_crossfit_splits": fold_count,\n    }\n    return calibrator, report\n\n\ndef compare_candidates(targets, base_neural, base_sparse, candidate, groups, seed=42, n_splits=5,\n                       min_gain=0.002, min_improved_folds=3):\n    """Tune on calibration-fold training rows, compare on excluded groups.\n\n    All supplied probabilities must originate from base/candidate models that\n    excluded this entire Training calibration set from model fitting and epoch\n    selection. These arrays must never contain official Validation samples.\n    Compare only the original baseline and fixed raw/logit mixture families.\n    The global procedure choice uses the resulting cross-fit scores, so its\n    selected score is a selection estimate, not an unbiased performance guarantee.\n    """\n    if not np.isfinite(min_gain) or min_gain < 0:\n        raise ValueError("min_gain must be finite and nonnegative")\n    if type(min_improved_folds) is not int or min_improved_folds < 1:\n        raise ValueError("min_improved_folds must be a positive integer")\n    truth, base, sparse, candidate = _validated_inputs(targets, base_neural, base_sparse, candidate)\n    baseline_predictions = np.zeros_like(truth)\n    mixture_predictions = {space: np.zeros_like(truth) for space in ("raw", "logit")}\n    seen = np.zeros(len(truth), dtype=np.int64)\n    fold_counts = []\n    fold_gains = {space: [] for space in ("raw", "logit")}\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        baseline_calibrator = _fit_baseline(truth[train], base[train], sparse[train])\n        baseline_predictions[test] = (\n            _apply_baseline(base[test], sparse[test], baseline_calibrator)\n            >= np.asarray(baseline_calibrator["thresholds"])\n        )\n        fold_baseline_score = f1_score(truth[test], baseline_predictions[test], average="macro", zero_division=0)\n        for space, predictions in mixture_predictions.items():\n            mixture_calibrator = _fit_mixture(\n                truth[train], base[train], sparse[train], candidate[train], blend_space=space,\n            )\n            predictions[test] = (\n                apply_selected(base[test], sparse[test], candidate[test], mixture_calibrator)\n                >= np.asarray(mixture_calibrator["thresholds"])\n            )\n            fold_gains[space].append(float(f1_score(truth[test], predictions[test], average="macro", zero_division=0)\n                                          - fold_baseline_score))\n        np.add.at(seen, test, 1)\n        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})\n    if not np.all(seen == 1):\n        raise RuntimeError("every calibration sample must be scored exactly once")\n    baseline_score = float(f1_score(truth, baseline_predictions, average="macro", zero_division=0))\n    mixture_scores = {\n        space: float(f1_score(truth, predictions, average="macro", zero_division=0))\n        for space, predictions in mixture_predictions.items()\n    }\n    # A single global family choice avoids per-symptom selection on scored folds.\n    candidate_space = "logit" if mixture_scores["logit"] > mixture_scores["raw"] else "raw"\n    candidate_score = mixture_scores[candidate_space]\n    candidate_predictions = mixture_predictions[candidate_space]\n    baseline_final = _with_zero_candidate(_fit_baseline(truth, base, sparse))\n    gain = candidate_score - baseline_score\n    # Never silently relax an explicitly requested adoption guard.\n    required_folds = min_improved_folds\n    improved_folds = sum(value > 0 for value in fold_gains[candidate_space])\n    material = candidate_score > baseline_score and gain >= min_gain\n    consistent = improved_folds >= required_folds\n    use_candidate = material and consistent\n    selected = (_fit_mixture(truth, base, sparse, candidate, blend_space=candidate_space)\n                if use_candidate else baseline_final)\n    selected_predictions = candidate_predictions if use_candidate else baseline_predictions\n    fitted = apply_selected(base, sparse, candidate, selected) >= np.asarray(selected["thresholds"])\n    report = {\n        "score_scope": "Training-only calibration. Procedure selection reuses cross-fit scores; selected score is a selection estimate, not an unbiased guarantee or official Validation.",\n        "baseline_crossfit_macro_f1": baseline_score,\n        "candidate_crossfit_macro_f1": candidate_score,\n        "candidate_raw_crossfit_macro_f1": mixture_scores["raw"],\n        "candidate_logit_crossfit_macro_f1": mixture_scores["logit"],\n        "candidate_blend_space": candidate_space,\n        "selected_blend_space": selected["blend_space"],\n        "selected_crossfit_macro_f1": candidate_score if use_candidate else baseline_score,\n        "selected_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),\n        "selected_name": "candidate_mixture" if use_candidate else "ver4_baseline",\n        "fallback_reason": (None if use_candidate else\n                            "candidate_gain_below_minimum" if not material else "candidate_gain_not_consistent_across_folds"),\n        "selection_gain": float(gain), "minimum_gain": float(min_gain),\n        "candidate_improved_folds": improved_folds, "minimum_improved_folds": required_folds,\n        "candidate_fold_gains": fold_gains[candidate_space], "all_candidate_fold_gains": fold_gains,\n        "per_symptom_crossfit": {\n            "baseline": _metrics(truth, baseline_predictions),\n            "candidate": _metrics(truth, candidate_predictions),\n            "candidate_raw": _metrics(truth, mixture_predictions["raw"]),\n            "candidate_logit": _metrics(truth, mixture_predictions["logit"]),\n            "selected": _metrics(truth, selected_predictions),\n        },\n        "calibration_samples": int(len(truth)),\n        "calibration_groups": int(len(np.unique(groups))),\n        "crossfit_splits": len(fold_counts),\n        "crossfit_fold_counts": fold_counts,\n        "selection_rule": "Compare fixed raw/logit families globally; ties prefer raw. Require strictly positive aggregate gain at least minimum_gain and improvements on minimum_improved_folds; otherwise use ver4. Training-only selection guard, not a statistical significance test or Validation guarantee.",\n    }\n    return selected, report\n', 'data.py': '"""Text-only turn boundaries for the independent ver9 encoder.\n\nThe shared source loader has already normalized each utterance and joined the\nturns with newlines. This representation deliberately accepts text strings,\nnever annotations, so labels and speaker metadata cannot become features.\n"""\nfrom __future__ import annotations\n\nimport unicodedata\n\n\ndef candidate_texts(texts, tokenizer):\n    """Insert the tokenizer\'s existing SEP token between nonempty turns.\n\n    No vocabulary token is added and no maximum dialogue length is imposed.\n    An empty dialogue remains empty and obtains the tokenizer\'s normal outer\n    special tokens when it is tokenized.\n    """\n    separator = getattr(tokenizer, "sep_token", None)\n    if not isinstance(separator, str) or not separator:\n        raise ValueError("The candidate tokenizer must define an existing sep_token")\n    if isinstance(texts, str):\n        raise TypeError("texts must be a sequence of dialogue strings")\n    rendered = []\n    for text in texts:\n        if not isinstance(text, str):\n            raise TypeError("Candidate inputs must be text strings, never annotations")\n        turns = [" ".join(unicodedata.normalize("NFC", turn).split()) for turn in text.split("\\n")]\n        rendered.append(f" {separator} ".join(turn for turn in turns if turn))\n    return rendered\n', 'model.py': '"""Submission entry point for the DeBERTa dialogue classifier architecture."""\nif __package__:\n    from .neural import DialogueClassifier\nelse:\n    from neural import DialogueClassifier\n\n__all__ = ["DialogueClassifier"]\n', 'neural.py': '"""Full Korean DeBERTa training with complete dialogue windows and bounded epochs.\n\nUnweighted BCE trains the entire public encoder plus CLS/mean window pooling.\nOnly grouped Training-stop cross-fit F1 selects an epoch. A wall-clock cutoff\ncan abandon a partial epoch without publishing its optimizer or model state.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport tempfile\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup\n\nif __package__:\n    from .selection import StopSelector, better\n    from .ver4_runtime.neural import NUM_LABELS, DialogueCollator, DialogueClassifier, _device, _loader, _optimizer, _targets, tokenize_dialogues\nelse:\n    from selection import StopSelector, better\n    from ver4_runtime.neural import NUM_LABELS, DialogueCollator, DialogueClassifier, _device, _loader, _optimizer, _targets, tokenize_dialogues\n\n\nMODEL_FORMAT = "mission3_ver9_neural"\nLAST_FORMAT = "mission3_ver9_last"\nDEFAULTS = {\n    "model_name": "team-lucid/deberta-v3-base-korean",\n    "max_length": 512, "stride": 64, "max_epochs": 3, "patience": 2,\n    "stop_crossfit_splits": 3, "learning_rate": 2e-5, "head_learning_rate": 1e-4,\n    "dropout": 0.1, "batch_size": 4, "window_batch_size": 4,\n    "gradient_accumulation_steps": 8, "inference_batch_size": 8,\n    "gradient_checkpointing": True, "device": "cuda", "log_every": 100,\n    "layerwise_lr_decay": 0.95, "weight_decay": 0.01, "warmup_ratio": 0.1,\n}\n\n\ndef native_bf16_supported():\n    if not torch.cuda.is_available():\n        return False\n    try:\n        return bool(torch.cuda.is_bf16_supported(including_emulation=False))\n    except TypeError:\n        return torch.cuda.get_device_capability()[0] >= 8\n\n\ndef _amp(device):\n    return torch.amp.autocast(device_type=device.type,\n                              dtype=torch.bfloat16 if device.type == "cuda" and native_bf16_supported() else torch.float16,\n                              enabled=device.type == "cuda")\n\n\ndef execution_identity(device):\n    """Exact epoch resume requires the same numerical execution environment."""\n    device = torch.device(device)\n    identity = {"device_type": device.type, "torch": str(torch.__version__),\n                "transformers": importlib.metadata.version("transformers"),\n                "cuda_version": torch.version.cuda,\n                "amp_dtype": "float32" if device.type != "cuda" else "bfloat16" if native_bf16_supported() else "float16",\n                "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),\n                "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,\n                "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32}\n    if device.type == "cuda":\n        identity.update(gpu=torch.cuda.get_device_name(device),\n                        capability=list(torch.cuda.get_device_capability(device)),\n                        cuda_device_count=torch.cuda.device_count())\n    else:\n        identity["cpu_threads"] = torch.get_num_threads()\n    return identity\n\n\ndef _settings(config):\n    settings = {name: config.get(name, value) for name, value in DEFAULTS.items()}\n    for name in ("max_epochs", "patience", "stop_crossfit_splits", "batch_size",\n                 "window_batch_size", "gradient_accumulation_steps", "inference_batch_size", "max_length", "stride", "log_every"):\n        minimum = 0 if name in ("max_epochs", "stride", "log_every") else 2 if name == "stop_crossfit_splits" else 1\n        if isinstance(settings[name], bool) or not isinstance(settings[name], int) or settings[name] < minimum:\n            raise ValueError(f"{name} must be an integer >= {minimum}")\n    if settings["max_epochs"] > 3:\n        raise ValueError("ver9 is bounded to max_epochs <= 3")\n    if settings["max_length"] > 512 or settings["max_length"] < 3 or settings["stride"] >= settings["max_length"] - 2:\n        raise ValueError("max_length must be in [3, 512]; stride must fit the semantic token window")\n    for name in ("learning_rate", "head_learning_rate", "layerwise_lr_decay"):\n        if isinstance(settings[name], bool) or not math.isfinite(float(settings[name])) or float(settings[name]) <= 0:\n            raise ValueError(f"{name} must be positive and finite")\n    if not 0 < settings["layerwise_lr_decay"] <= 1:\n        raise ValueError("layerwise_lr_decay must be in (0, 1]")\n    if not 0 <= float(settings["dropout"]) < 1 or not 0 <= float(settings["warmup_ratio"]) < 1:\n        raise ValueError("dropout/warmup_ratio must lie in [0, 1)")\n    if not math.isfinite(float(settings["weight_decay"])) or settings["weight_decay"] < 0:\n        raise ValueError("weight_decay must be nonnegative and finite")\n    if not isinstance(settings["gradient_checkpointing"], bool):\n        raise ValueError("gradient_checkpointing must be a boolean")\n    if not isinstance(settings["model_name"], str) or not settings["model_name"]:\n        raise ValueError("model_name must be nonempty")\n    return settings\n\n\ndef _configure_training(model, settings):\n    model.requires_grad_(True)\n    if not all(parameter.requires_grad for parameter in model.encoder.parameters()):\n        raise RuntimeError("The complete ver9 encoder must train from epoch one")\n    if settings["gradient_checkpointing"]:\n        model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    if hasattr(model.encoder.config, "use_cache"):\n        model.encoder.config.use_cache = False\n\n\ndef _read_checkpoint(path):\n    return torch.load(Path(path), map_location="cpu", weights_only=True, mmap=True)\n\n\ndef _encoder(configuration):\n    values = dict(configuration)\n    return AutoModel.from_config(AutoConfig.for_model(values.pop("model_type"), **values))\n\n\ndef _atomic_save(value, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    os.close(descriptor)\n    try:\n        torch.save(value, temporary)\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _rng_state(generator):\n    numpy_state = np.random.get_state()\n    return {"python": random.getstate(),\n            "numpy": {"name": numpy_state[0], "keys": numpy_state[1].tolist(), "position": int(numpy_state[2]),\n                      "has_gauss": int(numpy_state[3]), "cached_gaussian": float(numpy_state[4])},\n            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n            "loader": generator.get_state()}\n\n\ndef _restore_rng(state, generator):\n    random.setstate(state["python"])\n    values = state["numpy"]\n    np.random.set_state((values["name"], np.asarray(values["keys"], dtype=np.uint32), values["position"],\n                         values["has_gauss"], values["cached_gaussian"]))\n    torch.set_rng_state(state["torch"])\n    if state["cuda"]:\n        if not torch.cuda.is_available() or len(state["cuda"]) != torch.cuda.device_count():\n            raise ValueError("CUDA device count changed; exact epoch resume is unavailable")\n        torch.cuda.set_rng_state_all(state["cuda"])\n    generator.set_state(state["loader"])\n\n\ndef _seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef _cpu_state(module):\n    return {key: value.detach().cpu().clone() for key, value in module.state_dict().items()}\n\n\ndef _check_identity(state, signature, fingerprint, execution):\n    if state.get("run_signature") != signature:\n        raise ValueError("Checkpoint run signature mismatch; use a new output directory")\n    if state.get("execution") != execution:\n        raise ValueError("Exact epoch resume requires the same execution environment, GPU and precision; use a new output directory")\n    if state.get("configuration_fingerprint") != fingerprint:\n        raise ValueError("Checkpoint config fingerprint mismatch; use a new output directory")\n\n\nclass BudgetDeadlineExceeded(RuntimeError):\n    """The caller\'s absolute wall-clock training cutoff has been reached."""\n\n\ndef _check_deadline(deadline_epoch, reserve_seconds=0):\n    if deadline_epoch is not None and time.time() + reserve_seconds >= deadline_epoch:\n        raise BudgetDeadlineExceeded("Training phase deadline reached")\n\n\n@torch.inference_mode()\ndef _predict(model, loader, device, deadline_epoch=None):\n    model.eval()\n    probabilities = []\n    for batch in loader:\n        _check_deadline(deadline_epoch)\n        batch.pop("labels", None)\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with _amp(device):\n            logits = model(**batch)\n        if not torch.isfinite(logits).all():\n            raise FloatingPointError("Prediction logits contain nonfinite values")\n        probabilities.append(logits.float().sigmoid().cpu().numpy())\n        _check_deadline(deadline_epoch)\n    return np.concatenate(probabilities) if probabilities else np.empty((0, NUM_LABELS), dtype=np.float32)\n\n\ndef _from_saved(checkpoint, config):\n    if checkpoint.get("format") not in (MODEL_FORMAT, LAST_FORMAT) or checkpoint.get("format_version") != 1:\n        raise ValueError("Unsupported ver9 neural checkpoint")\n    architecture = dict(checkpoint["architecture"])\n    architecture["window_batch_size"] = int(config.get("window_batch_size", architecture["window_batch_size"]))\n    model = DialogueClassifier(_encoder(checkpoint["encoder_config"]), **architecture)\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    return model\n\n\ndef predict_checkpoint(checkpoint_path, documents, tokenizer, config):\n    """Reconstruct solely from saved config and weights, fully offline."""\n    checkpoint = _read_checkpoint(checkpoint_path)\n    if checkpoint.get("format") != MODEL_FORMAT:\n        raise ValueError("Inference requires the selected best ver9 checkpoint")\n    device = _device(config)\n    model = _from_saved(checkpoint, config).to(device)\n    del checkpoint\n    probabilities = _predict(model, _loader(documents, None, tokenizer, config.get("inference_batch_size", 8)),\n                             device, deadline_epoch=config.get("deadline_epoch"))\n    if not np.isfinite(probabilities).all():\n        raise FloatingPointError("Inference produced nonfinite probabilities")\n    return probabilities.astype(np.float32, copy=False)\n\n\ndef _best_payload(last):\n    # Inference artifacts do not carry Adam states or a second model copy.\n    payload = {key: value for key, value in last.items() if key not in ("optimizer", "scheduler", "scaler", "rng")}\n    return {**payload, "format": MODEL_FORMAT, "epoch": last["best_epoch"],\n            "stop_ap": last["best_score"]["stop_macro_ap"],\n            "stop_macro_f1": last["best_score"]["stop_macro_f1"]}\n\n\ndef _summary(state):\n    """Keep committed metadata without retaining an extra CPU encoder/Adam copy."""\n    keys = ("best_epoch", "best_score", "history", "completed_epochs", "completed", "execution",\n            "initialization", "initial_evaluated")\n    return {key: state[key] for key in keys}\n\n\ndef _result(state, resumed, last_path, stopped_for_budget=False, budget_reason=None):\n    return {"best_epoch": state["best_epoch"], "best_stop_ap": state["best_score"]["stop_macro_ap"],\n            "best_stop_macro_f1": state["best_score"]["stop_macro_f1"], "history": state["history"],\n            "completed_epochs": state["completed_epochs"], "completed": state["completed"],\n            "resumed": resumed, "last_checkpoint": str(last_path), "execution": state["execution"],\n            "best_is_initial_baseline": state["best_epoch"] == 0,\n            "initial_evaluated": state["initial_evaluated"], "head_type": "cls_mean_window_attention",\n            "initialization": state["initialization"], "loss": "unweighted_bce",\n            "stopped_for_budget": bool(stopped_for_budget), "budget_reason": budget_reason,\n            "selection_scope": "Training fold stop: grouped threshold cross-fit macro-F1; AP secondary. "\n                               "Reused epoch selection, not model-level OOF or official Validation."}\n\n\ndef train_model(train_documents, train_targets, stop_documents, stop_targets, tokenizer, config,\n                checkpoint_path, seed, initial_model_dir, run_signature, stop_groups=None, deadline_epoch=None):\n    """Train/resume from local public weights, committing completed epochs only.\n\n    ``deadline_epoch`` is an absolute ``time.time()`` timestamp for this training\n    phase. It is intentionally excluded from the configuration identity: a new\n    Colab session may resume the exact last committed epoch with a new cutoff.\n    A partial epoch is abandoned in memory, never promoted to a completed one.\n    """\n    train_targets = _targets(train_targets, train_documents, "fit")\n    stop_targets = _targets(stop_targets, stop_documents, "stop")\n    if stop_groups is None or np.asarray(stop_groups).shape != (len(stop_documents),):\n        raise ValueError("stop_groups must match stop documents")\n    if not isinstance(run_signature, str) or not run_signature:\n        raise ValueError("run_signature must be a nonempty string")\n    if deadline_epoch is not None:\n        if isinstance(deadline_epoch, bool) or not math.isfinite(float(deadline_epoch)):\n            raise ValueError("deadline_epoch must be a finite wall-clock timestamp")\n        deadline_epoch = float(deadline_epoch)\n    settings = _settings(config)\n    device = _device(settings)\n    execution = execution_identity(device)\n    fingerprint_data = {"settings": settings, "seed": int(seed),\n                        "stop_groups": list(map(str, stop_groups)), "execution": execution}\n    identity_hash = hashlib.sha256(json.dumps(fingerprint_data, sort_keys=True).encode())\n    for documents, targets in ((train_documents, train_targets), (stop_documents, stop_targets)):\n        for document in documents:\n            identity_hash.update(json.dumps(document, sort_keys=True, separators=(",", ":")).encode())\n            identity_hash.update(b"\\n")\n        identity_hash.update(targets.tobytes())\n    fingerprint = identity_hash.hexdigest()\n    checkpoint_path = Path(checkpoint_path)\n    last_path = checkpoint_path.with_name(checkpoint_path.stem + "_last.ckpt")\n    last = _read_checkpoint(last_path) if last_path.exists() else None\n    best_saved = _read_checkpoint(checkpoint_path) if checkpoint_path.exists() else None\n    for saved in (last, best_saved):\n        if saved is not None:\n            _check_identity(saved, run_signature, fingerprint, execution)\n    if best_saved is not None and best_saved.get("format") != MODEL_FORMAT:\n        raise ValueError("Unsupported ver9 best checkpoint")\n    if last is not None:\n        if last.get("format") != LAST_FORMAT:\n            raise ValueError("Unsupported ver9 resume checkpoint")\n        best_matches = (best_saved is not None and best_saved.get("epoch") == last["best_epoch"] and\n                        best_saved.get("stop_macro_f1") == last["best_score"]["stop_macro_f1"] and\n                        best_saved.get("stop_ap") == last["best_score"]["stop_macro_ap"])\n        if not best_matches:\n            if last["best_epoch"] != last["completed_epochs"]:\n                raise ValueError("Best checkpoint is missing/inconsistent with completed epoch state")\n            _atomic_save(_best_payload(last), checkpoint_path)\n        if last["completed"]:\n            return _result(last, True, last_path)\n    elif best_saved is not None:\n        raise ValueError("Best checkpoint exists without resume state")\n    del best_saved\n    selector = StopSelector(stop_targets, stop_groups, seed=seed, n_splits=settings["stop_crossfit_splits"])\n    resumed = last is not None\n    _seed(seed)\n    if last is None:\n        directory = Path(initial_model_dir)\n        if not directory.is_dir():\n            raise ValueError("initial_model_dir must be an already downloaded local public model directory")\n        encoder = AutoModel.from_pretrained(str(directory), local_files_only=True,\n                                            trust_remote_code=False, use_safetensors=True)\n        architecture = {"num_labels": NUM_LABELS, "dropout": settings["dropout"],\n                        "window_batch_size": settings["window_batch_size"]}\n        _seed(seed + 17)\n        model = DialogueClassifier(encoder, **architecture)\n        metadata = {"encoder_config": encoder.config.to_dict(), "architecture": architecture,\n                    "model_name": settings["model_name"],\n                    "initialization": "local_public_encoder_and_fresh_cls_mean_window_attention_head"}\n    else:\n        model = _from_saved(last, settings)\n        metadata = {key: last[key] for key in ("encoder_config", "architecture", "model_name", "initialization")}\n    model.to(device)\n    _configure_training(model, settings)\n    loader = _loader(train_documents, train_targets, tokenizer, settings["batch_size"], shuffle=True, seed=seed)\n    stop_loader = _loader(stop_documents, None, tokenizer, settings["inference_batch_size"])\n    criterion = nn.BCEWithLogitsLoss()\n    optimizer = _optimizer(model, settings)\n    accumulation = settings["gradient_accumulation_steps"]\n    total_steps = math.ceil(len(loader) / accumulation) * max(1, settings["max_epochs"])\n    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * settings["warmup_ratio"]), total_steps)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not native_bf16_supported())\n    if last is None:\n        best_score = {"stop_macro_f1": None, "stop_macro_ap": None}\n        history, completed, best_epoch, stale, initial_evaluated = [], 0, 0, 0, False\n    else:\n        optimizer.load_state_dict(last["optimizer"])\n        scheduler.load_state_dict(last["scheduler"])\n        scaler.load_state_dict(last["scaler"])\n        _restore_rng(last["rng"], loader.generator)\n        best_score, best_epoch, stale = last["best_score"], last["best_epoch"], last["stale_epochs"]\n        history, completed = list(last["history"]), last["completed_epochs"]\n        initial_evaluated = last["initial_evaluated"]\n    del last\n\n    def commit(epoch, done, improved):\n        state = {"format": LAST_FORMAT, "format_version": 1, **metadata, "model_state": _cpu_state(model),\n                 "execution": execution, "seed": int(seed), "run_signature": run_signature,\n                 "configuration_fingerprint": fingerprint, "settings": settings,\n                 "completed_epochs": epoch, "completed": done, "best_epoch": best_epoch,\n                 "best_score": best_score, "stale_epochs": stale, "history": list(history),\n                 "initial_evaluated": initial_evaluated, "loss": "unweighted_bce",\n                 "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),\n                 "scaler": scaler.state_dict(), "rng": _rng_state(loader.generator)}\n        # Publish last first so an interrupted best write is repairable whenever\n        # this newly committed epoch was the selected improvement.\n        _atomic_save(state, last_path)\n        if improved:\n            _atomic_save(_best_payload(state), checkpoint_path)\n        return _summary(state)\n\n    if resumed:\n        committed_state = {"best_epoch": best_epoch, "best_score": best_score, "history": list(history),\n                           "completed_epochs": completed, "completed": False, "execution": execution,\n                           "initialization": metadata["initialization"], "initial_evaluated": initial_evaluated}\n    else:\n        # Even if the first stop pass hits the cutoff, publish clearly marked\n        # untrained weights and a resumable pre-evaluation RNG state.\n        committed_state = commit(0, False, True)\n    if not initial_evaluated:\n        started = time.monotonic()\n        try:\n            best_score = selector.score(_predict(model, stop_loader, device, deadline_epoch))\n            _check_deadline(deadline_epoch)\n        except BudgetDeadlineExceeded:\n            return _result(committed_state, resumed, last_path, True, "initial_stop_evaluation")\n        history = [{"epoch": 0, "phase": "initial_public_encoder", "train_loss": None,\n                    **best_score, "minutes": (time.monotonic() - started) / 60}]\n        initial_evaluated = True\n        committed_state = commit(0, settings["max_epochs"] == 0, True)\n    if settings["max_epochs"] == 0:\n        return _result(committed_state, resumed, last_path)\n\n    for epoch in range(completed + 1, settings["max_epochs"] + 1):\n        # The slowest measured completed epoch includes training and stop\n        # evaluation. Keep 120 s for checkpoint publication; never start another\n        # epoch merely because some wall time remains.\n        measured = [row["minutes"] * 60 for row in history if row["epoch"] > 0]\n        estimated_next = max(measured, default=0.0) * 1.10\n        try:\n            _check_deadline(deadline_epoch, reserve_seconds=estimated_next + 120)\n        except BudgetDeadlineExceeded:\n            return _result(committed_state, resumed, last_path, True, "insufficient_time_for_next_epoch")\n        started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum, count = 0.0, 0\n        try:\n            for step, batch in enumerate(loader):\n                _check_deadline(deadline_epoch, reserve_seconds=120)\n                labels = batch.pop("labels").to(device)\n                batch = {key: value.to(device) for key, value in batch.items()}\n                group_start = (step // accumulation) * accumulation\n                group_documents = min(accumulation * loader.batch_size,\n                                      len(train_documents) - group_start * loader.batch_size)\n                with _amp(device):\n                    loss = criterion(model(**batch).float(), labels)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError(f"Nonfinite training loss at epoch={epoch}, batch={step}")\n                scaler.scale(loss * len(labels) / group_documents).backward()\n                loss_sum += float(loss.detach()) * len(labels)\n                count += len(labels)\n                if (step + 1) % accumulation == 0 or step + 1 == len(loader):\n                    scaler.unscale_(optimizer)\n                    nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=not scaler.is_enabled())\n                    old_scale = scaler.get_scale()\n                    scaler.step(optimizer)\n                    scaler.update()\n                    if scaler.get_scale() >= old_scale:\n                        scheduler.step()\n                    optimizer.zero_grad(set_to_none=True)\n                _check_deadline(deadline_epoch, reserve_seconds=120)\n                if settings["log_every"] > 0 and (step + 1) % settings["log_every"] == 0:\n                    print(f"V9 seed={seed} epoch={epoch} batch={step + 1}/{len(loader)} "\n                          f"loss={loss_sum / count:.5f}", flush=True)\n            evaluation_cutoff = None if deadline_epoch is None else deadline_epoch - 120\n            score = selector.score(_predict(model, stop_loader, device, evaluation_cutoff))\n            _check_deadline(deadline_epoch, reserve_seconds=120)\n        except BudgetDeadlineExceeded:\n            # The model/optimizer may contain partial updates. Do not call\n            # commit here: exact restart uses the previous complete epoch.\n            optimizer.zero_grad(set_to_none=True)\n            return _result(committed_state, resumed, last_path, True, "partial_epoch_abandoned")\n        history.append({"epoch": epoch, "phase": "full_encoder_finetune", "train_loss": loss_sum / count,\n                        **score, "minutes": (time.monotonic() - started) / 60})\n        improved = better(score, best_score)\n        if improved:\n            best_score, best_epoch, stale = score, epoch, 0\n        else:\n            stale += 1\n        done = epoch == settings["max_epochs"] or stale >= settings["patience"]\n        committed_state = commit(epoch, done, improved)\n        print(f"V9 epoch={epoch}/{settings[\'max_epochs\']} loss={loss_sum / count:.5f} "\n              f"stop_macro_F1={score[\'stop_macro_f1\']:.6f} stop_AP={score[\'stop_macro_ap\']:.6f} "\n              f"best_epoch={best_epoch}", flush=True)\n        if done:\n            return _result(committed_state, resumed, last_path)\n    raise RuntimeError("Resume checkpoint marked incomplete after the configured epoch budget")\n\n', 'pipeline.py': '"""Training-only DeBERTa/NB-LR candidate and portable guarded baseline ensemble."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport time\n\nimport joblib\nimport numpy as np\nimport torch\nfrom transformers import AutoTokenizer\n\nif __package__:\n    from . import assets, calibration, data, neural, pretrained, sparse\n    from .budget import TimeBudget, BudgetExceeded\n    from .prediction_cache import PredictionCache\n    from .ver4_runtime import pipeline as base_pipeline, neural as base_neural\n    from .ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, write_predictions\nelse:\n    import assets, calibration, data, neural, pretrained, sparse\n    from budget import TimeBudget, BudgetExceeded\n    from prediction_cache import PredictionCache\n    from ver4_runtime import pipeline as base_pipeline, neural as base_neural\n    from ver4_runtime.data import TARGET_SYMPTOMS, load_dataset, write_predictions\n\nDEFAULT_CONFIG = {\n    **neural.DEFAULTS, "seed": 42, "max_models": 2, "require_cuda": True,\n    "min_calibration_gain": 0.002, "min_calibration_improved_folds": 3,\n    "final_reserve_seconds": 1500, "calibration_reserve_seconds": 600,\n    "keep_completed_resume_states": False,\n}\nCANDIDATE_NEURAL_WEIGHT = 0.75  # Declared before any calibration/Validation results.\nMANIFEST = "mission3_ver9_model.ckpt"  # JSON ensemble manifest; weights are separate .ckpt files.\nRUNTIME_FILES = (\n    "__init__.py", "assets.py", "budget.py", "bootstrap.py", "calibration.py", "data.py",\n    "model.py", "neural.py", "pipeline.py", "prediction_cache.py", "pretrained.py",\n    "selection.py", "sparse.py", "train.py", "inference.py", "run.py",\n    "requirements.txt", "README.md", "RESEARCH.md",\n)\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef export_code(output_dir):\n    source, destination = Path(__file__).parent, Path(output_dir) / "submission"\n    destination.mkdir(parents=True, exist_ok=True)\n    for name in RUNTIME_FILES:\n        shutil.copyfile(source / name, destination / name)\n    for path in source.joinpath("ver4_runtime").iterdir():\n        if path.is_file():\n            target = destination / "ver4_runtime" / path.name\n            target.parent.mkdir(exist_ok=True)\n            shutil.copyfile(path, target)\n\n\ndef _safe_file(root, name):\n    path = Path(name)\n    if path.is_absolute() or ".." in path.parts or "\\\\" in str(name):\n        raise ValueError("Model paths must remain inside the portable result folder")\n    result = root / path\n    if not result.resolve().is_relative_to(root.resolve()) or not result.exists():\n        raise ValueError(f"Missing or unsafe model artifact: {name}")\n    return result\n\n\ndef load_manifest(checkpoint_path):\n    path = Path(checkpoint_path).resolve()\n    manifest = json.loads(path.read_text(encoding="utf-8"))\n    if manifest.get("format") != "mission3_ver9" or manifest.get("labels") != list(TARGET_SYMPTOMS):\n        raise ValueError("Unsupported ver9 manifest or label order")\n    base_path = _safe_file(path.parent, manifest["base_manifest"])\n    base = base_pipeline.load_manifest(base_path)\n    required = {manifest["base_manifest"], *[str(Path(manifest["base_manifest"]).parent / n) for n in [*base["folds"], base["sparse"]]]}\n    required.update(p.relative_to(path.parent).as_posix() for p in (base_path.parent / base["tokenizer"]).rglob("*") if p.is_file())\n    if manifest["candidate_models"]:\n        required.update(manifest["candidate_models"])\n        required.add(manifest["candidate_sparse"])\n        tokenizer = _safe_file(path.parent, manifest["tokenizer"])\n        if not tokenizer.is_dir() or not (tokenizer / "tokenizer.json").is_file():\n            raise ValueError("Missing candidate fast tokenizer")\n        required.update(p.relative_to(path.parent).as_posix() for p in tokenizer.rglob("*") if p.is_file())\n    elif any(manifest["selection"]["candidate_weights"]):\n        raise ValueError("Selected candidate has no trained checkpoint")\n    hashes = manifest.get("artifact_sha256")\n    if not isinstance(hashes, dict) or not required <= hashes.keys():\n        raise ValueError("Manifest must cover all model artifacts with SHA256 checksums")\n    for name, digest in hashes.items():\n        if assets._sha256(_safe_file(path.parent, name)) != digest:\n            raise ValueError(f"Model artifact checksum mismatch: {name}")\n    return manifest\n\n\ndef _config(supplied):\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown ver9 options: {sorted(unknown)}")\n    config = {**DEFAULT_CONFIG, **supplied}\n    for name in ("final_reserve_seconds", "calibration_reserve_seconds", "min_calibration_gain"):\n        if isinstance(config[name], bool) or not isinstance(config[name], (int, float)) or not math.isfinite(config[name]):\n            raise ValueError(f"{name} must be a finite number")\n    for name in ("seed", "min_calibration_improved_folds"):\n        if type(config[name]) is not int or config[name] < 0:\n            raise ValueError(f"{name} must be a nonnegative integer")\n    if type(config["max_models"]) is not int or not 1 <= config["max_models"] <= 2:\n        raise ValueError("max_models must be 1 or 2")\n    if config["model_name"] != pretrained.MODEL_ID:\n        raise ValueError("ver9 requires the pinned Korean DeBERTa-v3 model")\n    if config["final_reserve_seconds"] < 1500 or config["calibration_reserve_seconds"] < 600:\n        raise ValueError("Reserve at least 25 minutes for final inference and 10 for calibration")\n    if config["min_calibration_gain"] < .002 or config["min_calibration_improved_folds"] < 3:\n        raise ValueError("Do not relax the predeclared adoption guard")\n    neural._settings(config)\n    return config\n\n\ndef _identity(config, base_manifest, base_dir, pretrained_metadata):\n    source = Path(__file__).parent\n    code = {p.relative_to(source).as_posix(): assets._sha256(p)\n            for p in [*[source / name for name in RUNTIME_FILES if name.endswith(".py")],\n                      *sorted((source / "ver4_runtime").glob("*.py"))]}\n    files = [*base_manifest["folds"], base_manifest["sparse"], "mission3_ver4_model.ckpt"]\n    files += [p.relative_to(base_dir).as_posix() for p in (base_dir / base_manifest["tokenizer"]).rglob("*") if p.is_file()]\n    hashes = {name: assets._sha256(base_dir / name) for name in files}\n    identity = {"config": config, "base_signature": base_manifest["signature"], "base_artifact_sha256": hashes,\n                "splits_sha256": assets._sha256(base_dir / "splits.json"),\n                "calibration_sha256": assets._sha256(base_dir / "calibration_predictions.npz"),\n                "pretrained": {k: pretrained_metadata[k] for k in ("model_id", "revision", "files")},\n                "code": code, "versions": assets._versions(), "candidate_neural_weight": CANDIDATE_NEURAL_WEIGHT}\n    signature = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n    return signature, identity\n\n\ndef train_pipeline(train_label_dir, base_dir, output_dir, config=None, pretrained_dir=None,\n                   start_epoch=None, deadline_epoch=None, attempt_marker=None):\n    """No Validation path is accepted. All choices finish within Training partitions."""\n    started = time.time()\n    if (start_epoch is None) != (deadline_epoch is None):\n        raise ValueError("Pass the shared start_epoch and deadline_epoch together")\n    budget = TimeBudget(start_epoch=start_epoch, deadline_epoch=deadline_epoch)\n    config = _config(config or {})\n    if config["require_cuda"] and not torch.cuda.is_available():\n        raise RuntimeError("Colab GPU runtime is required; CPU training would exceed the budget")\n    base_dir, output_dir = Path(base_dir).resolve(), Path(output_dir).resolve()\n    if output_dir == base_dir or output_dir.is_relative_to(base_dir) or base_dir.is_relative_to(output_dir):\n        raise ValueError("Output must be separate from original ver4 inputs")\n    output_dir.mkdir(parents=True, exist_ok=True)\n    metadata = pretrained.validate_pretrained(pretrained_dir)\n    if metadata["model_id"] != pretrained.MODEL_ID or metadata["revision"] != pretrained.MODEL_REVISION:\n        raise ValueError("Unexpected public pretrained model/revision")\n    base, paths, texts, targets, fit, stop, cal, groups, base_probs, base_sparse = assets._read_base(train_label_dir, base_dir, 1)\n    signature, identity = _identity(config, base, base_dir, metadata)\n    identity_path = output_dir / "run_identity.json"\n    if identity_path.exists():\n        if json.loads(identity_path.read_text())["signature"] != signature:\n            raise ValueError("Output contains a different experiment; choose a new output directory")\n    else:\n        if (output_dir / MANIFEST).exists():\n            raise ValueError("Existing output lacks a verified run identity")\n        write_json(identity_path, {"signature": signature, **identity})\n    completed_path = output_dir / "training_metrics.json"\n    if completed_path.exists() and json.loads(completed_path.read_text()).get("complete"):\n        load_manifest(output_dir / MANIFEST)\n        if attempt_marker is not None:\n            write_json(attempt_marker, {"signature": signature, "output_dir": str(output_dir)})\n        print("Completed ver9 run verified; reusing frozen model selection.", flush=True)\n        return json.loads(completed_path.read_text())\n    export_code(output_dir)\n    budget.check(config["final_reserve_seconds"] + config["calibration_reserve_seconds"], phase="portable base preparation")\n    assets._export_base(base_dir, output_dir, base, identity["base_artifact_sha256"])\n    manifest = {"format": "mission3_ver9", "format_version": 1, "labels": list(TARGET_SYMPTOMS),\n                "signature": signature, "config": config, "base_manifest": "base/mission3_ver4_model.ckpt",\n                "selected_name": "ver4_baseline", "selection": {**base["calibrator"], "candidate_weights": [0.] * 9, "blend_space": "raw"},\n                "candidate_models": [], "candidate_neural_weight": CANDIDATE_NEURAL_WEIGHT,\n                "artifact_sha256": {"base/" + k: v for k, v in identity["base_artifact_sha256"].items()},\n                "status": "baseline_ready", "versions": assets._versions()}\n    # Atomic early fallback survives a later candidate timeout or exception.\n    write_json(output_dir / MANIFEST, manifest)\n    if attempt_marker is not None:\n        write_json(attempt_marker, {"signature": signature, "output_dir": str(output_dir)})\n    tokenizer = AutoTokenizer.from_pretrained(pretrained_dir, use_fast=True, local_files_only=True, trust_remote_code=False)\n    tokenizer.save_pretrained(output_dir / "tokenizer")\n    documents = neural.tokenize_dialogues(data.candidate_texts(texts, tokenizer), tokenizer, config["max_length"], config["stride"])\n    filenames = [paths[i].name for i in cal]\n    cal_documents = [documents[i] for i in cal]\n    splits = json.loads((base_dir / "splits.json").read_text(encoding="utf-8"))\n    lookup = {path.name: i for i, path in enumerate(paths)}\n    histories, candidate_models, cal_predictions = [], [], []\n    training_cutoff = budget.deadline_epoch - config["final_reserve_seconds"] - config["calibration_reserve_seconds"] - 20\n    first_duration = None\n    for fold_number in range(1, config["max_models"] + 1):\n        available = training_cutoff - time.time()\n        # The optional second fold gets a full measured run allowance + 20% margin.\n        if available <= 120 or (fold_number > 1 and (first_duration is None or available < first_duration * 1.2 + 120)):\n            print(f"Fold {fold_number} skipped: preserving final inference/calibration time.", flush=True)\n            break\n        saved = splits["folds"][fold_number - 1]\n        fit = np.asarray([lookup[name] for name in saved["fit"]])\n        stop = np.asarray([lookup[name] for name in saved["stop"]])\n        checkpoint = output_dir / f"candidate_{fold_number}.ckpt"\n        fold_started = time.time()\n        history = neural.train_model([documents[i] for i in fit], targets[fit], [documents[i] for i in stop], targets[stop],\n                                     tokenizer, config, checkpoint, config["seed"] + fold_number * 1009,\n                                     pretrained_dir, signature, stop_groups=groups[stop], deadline_epoch=training_cutoff)\n        histories.append({"fold": fold_number, **history})\n        write_json(output_dir / "candidate_history.json", histories)\n        assets._release()\n        if history["best_epoch"] > 0:\n            prediction_config = {**config, "deadline_epoch": budget.deadline_epoch - config["final_reserve_seconds"] - 30}\n            probabilities = assets._cached_predictions(output_dir / f"cal_candidate_{fold_number}.npz", signature, filenames,\n                assets._sha256(checkpoint), lambda: neural.predict_checkpoint(checkpoint, cal_documents, tokenizer, prediction_config))\n            cal_predictions.append(probabilities)\n            candidate_models.append(checkpoint.name)\n            assets._release()\n        first_duration = time.time() - fold_started if first_duration is None else first_duration\n        if history["stopped_for_budget"]:\n            break\n    if not candidate_models:\n        report = {"complete": True, "status": "no_trained_candidate", "selected_name": "ver4_baseline", "histories": histories,\n                  "training_seconds": time.time() - started, "reason": "No trained epoch improved the initial head, or the time budget was exhausted."}\n        manifest["status"] = report["status"]\n        write_json(output_dir / MANIFEST, manifest)\n        write_json(completed_path, report)\n        return report\n    budget.check(config["final_reserve_seconds"] + 60, phase="NB-LR and Training calibration")\n    noncal = np.setdiff1d(np.arange(len(paths)), cal)\n    sparse_model = sparse.fit_model([texts[i] for i in noncal], targets[noncal], {"seed": config["seed"]})\n    temporary = output_dir / "nb_ratio_lr.joblib.tmp"\n    joblib.dump(sparse_model, temporary, compress=3)\n    temporary.replace(output_dir / "nb_ratio_lr.joblib")\n    sparse_probs = sparse.predict_model(sparse_model, [texts[i] for i in cal])\n    candidate = CANDIDATE_NEURAL_WEIGHT * np.mean(cal_predictions, axis=0) + (1 - CANDIDATE_NEURAL_WEIGHT) * sparse_probs\n    selection, comparison = calibration.compare_candidates(targets[cal], base_probs, base_sparse, candidate, groups[cal],\n        seed=config["seed"] + 2, min_gain=config["min_calibration_gain"], min_improved_folds=config["min_calibration_improved_folds"])\n    standalone, diagnostic = calibration.fit_candidate_only_crossfit(targets[cal], candidate, groups[cal], seed=config["seed"] + 2)\n    # The fallback is the exact existing submitted ver4 calibrator, not a re-tuned variant.\n    if comparison["selected_name"] == "ver4_baseline":\n        selection = manifest["selection"]\n    manifest.update(status="selection_frozen", selected_name=comparison["selected_name"], selection=selection,\n                    candidate_models=candidate_models, candidate_sparse="nb_ratio_lr.joblib", tokenizer="tokenizer",\n                    candidate_calibrator=standalone)\n    for name in [*candidate_models, "nb_ratio_lr.joblib", *[p.relative_to(output_dir).as_posix() for p in (output_dir / "tokenizer").rglob("*") if p.is_file()]]:\n        manifest["artifact_sha256"][name] = assets._sha256(output_dir / name)\n    write_json(output_dir / "calibration_comparison.json", comparison)\n    write_json(output_dir / "candidate_calibration_metrics.json", diagnostic)\n    with (output_dir / "calibration_predictions.npz.tmp").open("wb") as stream:\n        np.savez_compressed(stream, filenames=np.asarray(filenames), targets=targets[cal], candidate=candidate,\n                            candidate_neural=np.mean(cal_predictions, axis=0), nb_ratio_lr=sparse_probs)\n    (output_dir / "calibration_predictions.npz.tmp").replace(output_dir / "calibration_predictions.npz")\n    write_json(output_dir / MANIFEST, manifest)\n    report = {"complete": True, "status": "selection_frozen", "selected_name": comparison["selected_name"], "histories": histories,\n              "candidate_models": candidate_models, "candidate_neural_weight": CANDIDATE_NEURAL_WEIGHT,\n              "calibration": comparison, "candidate_only": diagnostic, "training_seconds": time.time() - started,\n              "total_elapsed_seconds": time.time() - budget.start_epoch, "remaining_seconds": budget.remaining_seconds(),\n              "official_validation_used_for_selection": False}\n    write_json(completed_path, report)\n    if not config["keep_completed_resume_states"]:\n        for history in histories:\n            # Only remove this experiment\'s optimizer states after final selection is durable.\n            if history["completed"]:\n                (output_dir / f"candidate_{history[\'fold\']}_last.ckpt").unlink(missing_ok=True)\n    return report\n\n\ndef _raw_base(root, base, paths, texts, config, cache_dir):\n    tokenizer_dir = root / base["tokenizer"]\n    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True, local_files_only=True)\n    documents = base_neural.tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    cache = PredictionCache(cache_dir, paths, texts, tokenizer_dir, TARGET_SYMPTOMS,\n                            neural.execution_identity(config["device"])) if cache_dir else None\n    sources = [Path(base_pipeline.__file__), Path(base_neural.__file__)]\n    outputs = []\n    for name in base["folds"]:\n        print("Base prediction:", name, flush=True)\n        compute = lambda name=name: base_neural.predict_checkpoint(root / name, documents, tokenizer, config)\n        outputs.append(cache.get("ver4-neural", root / name, config, sources, compute) if cache else compute())\n        assets._release()\n    return np.mean(np.asarray(outputs, dtype=np.float64), axis=0), base_pipeline.predict_sparse(joblib.load(root / base["sparse"]), texts)\n\n\ndef predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None, cache_dir=None, candidate_output=None):\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = load_manifest(checkpoint_path)\n    root = checkpoint_path.parent\n    base_path = _safe_file(root, manifest["base_manifest"])\n    base = base_pipeline.load_manifest(base_path)\n    assets._check_versions(base)\n    config = {**base["config"], "device": device or ("cuda" if torch.cuda.is_available() else "cpu")}\n    if batch_size is not None:\n        config["inference_batch_size"] = batch_size\n    # This loader ignores symptom targets and all metadata, even if present.\n    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)\n    base_probs, base_sparse = _raw_base(base_path.parent, base, paths, texts, config, cache_dir)\n    candidate = None\n    if any(manifest["selection"]["candidate_weights"]) or candidate_output:\n        if not manifest["candidate_models"]:\n            if any(manifest["selection"]["candidate_weights"]):\n                raise ValueError("Selected candidate has no trained checkpoint")\n            print("No trained candidate; standalone diagnostic skipped.", flush=True)\n        else:\n            tokenizer = AutoTokenizer.from_pretrained(_safe_file(root, manifest["tokenizer"]), use_fast=True, local_files_only=True)\n            candidate_config = {**manifest["config"], "device": config["device"], "inference_batch_size": config["inference_batch_size"]}\n            documents = neural.tokenize_dialogues(data.candidate_texts(texts, tokenizer), tokenizer,\n                                                 candidate_config["max_length"], candidate_config["stride"])\n            outputs = []\n            for name in manifest["candidate_models"]:\n                outputs.append(neural.predict_checkpoint(_safe_file(root, name), documents, tokenizer, candidate_config))\n                assets._release()\n            sparse_probs = sparse.predict_model(joblib.load(_safe_file(root, manifest["candidate_sparse"])), texts)\n            weight = manifest["candidate_neural_weight"]\n            candidate = weight * np.mean(outputs, axis=0) + (1 - weight) * sparse_probs\n    probabilities = calibration.apply_selected(base_probs, base_sparse, candidate, manifest["selection"])\n    # Publish complete CSVs atomically; a deadline cannot leave a truncated final CSV.\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output.with_name(output.name + ".tmp")\n    predictions = write_predictions(paths, probabilities, manifest["selection"]["thresholds"], temporary)\n    temporary.replace(output)\n    if candidate_output and candidate is not None:\n        destination = Path(candidate_output)\n        temporary = destination.with_name(destination.name + ".tmp")\n        write_predictions(paths, candidate, manifest["candidate_calibrator"]["thresholds"], temporary)\n        temporary.replace(destination)\n    print(f"Saved {len(paths):,} predictions: {output}", flush=True)\n    return paths, probabilities, predictions\n\n\nevaluate_csv = base_pipeline.evaluate_csv\n', 'prediction_cache.py': '"""Reuse raw inference probabilities across experiments without reading labels.\n\nKeys bind ordered inputs, weights, tokenizer, inference code and execution\nsettings. Calibration/thresholds are deliberately excluded: they are applied\nafter loading raw probabilities. A missing or damaged cache is recomputed.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\nimport tempfile\nfrom zipfile import BadZipFile\n\nimport numpy as np\n\n\ndef _file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _json(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n\n\nclass PredictionCache:\n    def __init__(self, directory, paths, texts, tokenizer_dir, labels, execution):\n        self.directory = Path(directory).expanduser().resolve()\n        self.shape = (len(paths), len(labels))\n        if len(paths) != len(texts):\n            raise ValueError("Cache paths/texts length mismatch")\n        tokenizer_dir = Path(tokenizer_dir)\n        self.context = {\n            "format": "mission3_raw_probabilities_v1",\n            "inputs": hashlib.sha256(_json([\n                [Path(path).name, text] for path, text in zip(paths, texts)\n            ]).encode("utf-8")).hexdigest(),\n            "labels": list(labels), "execution": execution,\n            "tokenizer": {p.relative_to(tokenizer_dir).as_posix(): _file_hash(p)\n                          for p in sorted(tokenizer_dir.rglob("*")) if p.is_file()},\n        }\n\n    def _valid(self, values):\n        return (values.shape == self.shape and np.isfinite(values).all()\n                and np.all((values >= 0) & (values <= 1)))\n\n    def get(self, kind, checkpoint, config, sources, compute):\n        # Do not bind absolute paths: a copied identical bundle can reuse results.\n        identity = _json({\n            **self.context, "kind": kind, "weights": _file_hash(checkpoint),\n            "settings": {key: config[key] for key in\n                         ("device", "max_length", "stride", "window_batch_size", "inference_batch_size")},\n            "code": [_file_hash(path) for path in sources],\n        })\n        key = hashlib.sha256(identity.encode("utf-8")).hexdigest()\n        path = self.directory / (key + ".npz")\n        if path.is_file():\n            try:\n                with np.load(path, allow_pickle=False) as stored:\n                    values = np.array(stored["probabilities"], dtype=np.float64)\n                    checksum = hashlib.sha256(values.tobytes()).hexdigest()\n                    if (stored["identity"].item() == identity and self._valid(values)\n                            and stored["checksum"].item() == checksum):\n                        print(f"예측 캐시 재사용: {kind} / {Path(checkpoint).name}", flush=True)\n                        return values\n            except (OSError, ValueError, KeyError, EOFError, BadZipFile):\n                pass\n            print(f"예측 캐시 손상: {kind}, 다시 계산합니다.", flush=True)\n        values = np.asarray(compute(), dtype=np.float64)\n        if not self._valid(values):\n            raise ValueError("Invalid raw probabilities; cache was not written")\n        self.directory.mkdir(parents=True, exist_ok=True)\n        # Unique temporary files keep interrupted/concurrent writes out of reads.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(dir=self.directory, suffix=".tmp", delete=False) as stream:\n                temporary = Path(stream.name)\n                np.savez_compressed(stream, identity=np.asarray(identity), probabilities=values,\n                                    checksum=np.asarray(hashlib.sha256(values.tobytes()).hexdigest()))\n            temporary.replace(path)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n        print(f"예측 캐시 저장: {kind} / {Path(checkpoint).name}", flush=True)\n        return values\n', 'pretrained.py': '"""Prepare one pinned public encoder; training and inference only use local files."""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport tempfile\nimport uuid\n\nMODEL_ID = "team-lucid/deberta-v3-base-korean"\nMODEL_REVISION = "47a3b00828c5dc2b8b080c6738ef7a58ebd419d5"\nMANIFEST_NAME = "pretrained_manifest.json"\nFORMAT = "mission3_ver9_pretrained_v1"\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef validate_pretrained(directory) -> dict:\n    """Return provenance only after checking a complete, safe local artifact set.\n\n    This function never contacts Hugging Face and does not load model weights.\n    File hashes are returned under ``files[name][\'sha256\']`` for run identities.\n    """\n    root = Path(directory).expanduser().resolve()\n    marker = root / MANIFEST_NAME\n    if not marker.is_file() or marker.is_symlink():\n        raise ValueError(f"사전학습 모델 준비가 완료되지 않았습니다: {marker}")\n    try:\n        metadata = json.loads(marker.read_text(encoding="utf-8"))\n    except (OSError, ValueError) as error:\n        raise ValueError(f"잘못된 사전학습 모델 manifest: {marker}") from error\n    if (metadata.get("format") != FORMAT or metadata.get("complete") is not True\n            or not isinstance(metadata.get("model_id"), str)\n            or not re.fullmatch(r"[0-9a-f]{40}", str(metadata.get("revision", "")))):\n        raise ValueError("사전학습 모델 manifest 형식/고정 revision이 올바르지 않습니다.")\n    files = metadata.get("files")\n    if not isinstance(files, dict) or not files:\n        raise ValueError("사전학습 모델 파일 목록이 없습니다.")\n    required = {"config.json", "tokenizer_config.json", "tokenizer.json"}\n    if not required <= files.keys() or not ("model.safetensors" in files\n                                           or "model.safetensors.index.json" in files):\n        raise ValueError("encoder config/safetensors/fast tokenizer 파일이 빠졌습니다.")\n    actual = {p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()}\n    if actual != set(files) | {MANIFEST_NAME}:\n        raise ValueError("사전학습 모델의 실제 파일 목록과 manifest가 다릅니다.")\n    for name, expected in files.items():\n        relative = PurePosixPath(name)\n        if (relative.is_absolute() or ".." in relative.parts or "\\\\" in name\n                or relative.as_posix() != name or name == MANIFEST_NAME):\n            raise ValueError(f"안전하지 않은 사전학습 파일 경로: {name}")\n        path = root / name\n        if (path.is_symlink() or any(parent.is_symlink() for parent in path.parents if parent != root)\n                or not path.is_file() or not path.resolve().is_relative_to(root)):\n            raise ValueError(f"정상적인 로컬 사전학습 파일이 아닙니다: {name}")\n        if (not isinstance(expected, dict) or path.stat().st_size != expected.get("size")\n                or _sha256(path) != expected.get("sha256")):\n            raise ValueError(f"사전학습 파일 무결성 검사 실패: {name}")\n    return metadata\n\n\ndef prepare_pretrained(destination, model_id=MODEL_ID, revision=MODEL_REVISION) -> dict:\n    """Download a pinned encoder/tokenizer once and atomically publish local files.\n\n    Only this explicit preparation entry point may use the network. It receives no\n    competition texts or labels. A valid completed directory is reused offline.\n    A failed preparation leaves the existing destination untouched.\n    """\n    if not re.fullmatch(r"[0-9a-f]{40}", str(revision)):\n        raise ValueError("revision은 변경되지 않는 40자리 Hugging Face commit SHA여야 합니다.")\n    raw_destination = Path(destination).expanduser()\n    if raw_destination.is_symlink():\n        raise ValueError("사전학습 모델 저장 경로에 심볼릭 링크를 사용할 수 없습니다.")\n    root = raw_destination.resolve()\n    if root.exists():\n        try:\n            metadata = validate_pretrained(root)\n        except ValueError:\n            # Refuse to replace unrelated directories; our own corrupt bundle may be repaired.\n            marker = root / MANIFEST_NAME\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = {}\n            if previous.get("format") != FORMAT:\n                raise ValueError(f"완료 manifest가 없는 기존 폴더입니다. 비어 있는 새 경로를 사용하세요: {root}")\n        else:\n            if metadata["model_id"] != model_id or metadata["revision"] != revision:\n                raise ValueError("기존 사전학습 모델과 요청한 모델/revision이 다릅니다. 새 경로를 사용하세요.")\n            print(f"검증된 사전학습 모델 재사용: {root}", flush=True)\n            return metadata\n    root.parent.mkdir(parents=True, exist_ok=True)\n    staging = Path(tempfile.mkdtemp(prefix="." + root.name + ".preparing-", dir=root.parent))\n    backup = None\n    try:\n        from transformers import AutoModel, AutoTokenizer\n\n        print(f"공개 사전학습 encoder 다운로드: {model_id}@{revision}", flush=True)\n        options = {"revision": revision, "local_files_only": False, "trust_remote_code": False}\n        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, **options)\n        # Retain only the public encoder: ver9 performs no additional MLM/TAPT.\n        model = AutoModel.from_pretrained(model_id, use_safetensors=True, **options)\n        tokenizer.save_pretrained(staging)\n        model.save_pretrained(staging, safe_serialization=True)\n        del model, tokenizer\n        # Prove that the published form is self-contained and works offline.\n        check_model = AutoModel.from_pretrained(staging, local_files_only=True, trust_remote_code=False,\n                                               use_safetensors=True)\n        check_tokenizer = AutoTokenizer.from_pretrained(staging, local_files_only=True,\n                                                       trust_remote_code=False, use_fast=True)\n        if not check_tokenizer.is_fast:\n            raise ValueError("window 생성에 사용할 fast tokenizer가 필요합니다.")\n        del check_model, check_tokenizer\n        metadata = {\n            "format": FORMAT, "complete": True, "model_id": model_id, "revision": revision,\n            "prepared_at_utc": datetime.now(timezone.utc).isoformat(),\n            "files": {path.relative_to(staging).as_posix(): {"size": path.stat().st_size, "sha256": _sha256(path)}\n                      for path in sorted(staging.rglob("*")) if path.is_file()},\n        }\n        (staging / MANIFEST_NAME).write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\\n",\n                                           encoding="utf-8")\n        validate_pretrained(staging)\n        if root.exists():\n            backup = root.with_name("." + root.name + ".previous-" + uuid.uuid4().hex)\n            os.replace(root, backup)\n        try:\n            os.replace(staging, root)\n        except BaseException:\n            if backup is not None:\n                os.replace(backup, root)\n                backup = None\n            raise\n        if backup is not None:\n            shutil.rmtree(backup)\n            backup = None\n        print(f"사전학습 모델 준비 완료: {root}", flush=True)\n        return metadata\n    finally:\n        if staging.exists():\n            shutil.rmtree(staging)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--destination", required=True)\n    parser.add_argument("--model_id", default=MODEL_ID)\n    parser.add_argument("--revision", default=MODEL_REVISION)\n    args = parser.parse_args()\n    prepare_pretrained(args.destination, model_id=args.model_id, revision=args.revision)\n\n\nif __name__ == "__main__":\n    main()\n', 'selection.py': '"""Grouped threshold cross-fitting on the original fold\'s stopping split.\n\nThis is model/epoch selection on Training stop. It is not model-level OOF and\nits repeatedly selected score is not an unbiased final performance estimate.\nNo official Validation or model-calibration rows belong here.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import average_precision_score, f1_score\n\nif __package__:\n    from .ver4_runtime.calibration import _best_threshold, _grouped_splits\nelse:\n    from ver4_runtime.calibration import _best_threshold, _grouped_splits\n\n\nclass StopSelector:\n    def __init__(self, targets, groups, seed=44, n_splits=3):\n        self.targets = np.asarray(targets, dtype=np.float32)\n        if self.targets.ndim != 2 or not len(self.targets) or not np.isin(self.targets, [0, 1]).all():\n            raise ValueError("Stop targets must be a nonempty binary matrix")\n        self.groups = np.asarray(groups)\n        self.folds = list(_grouped_splits(self.targets, self.groups, seed, n_splits))\n\n    def score(self, probabilities):\n        probabilities = np.asarray(probabilities, dtype=np.float64)\n        if probabilities.shape != self.targets.shape or not np.isfinite(probabilities).all():\n            raise ValueError("Stop probabilities must be finite and match targets")\n        if np.any((probabilities < 0) | (probabilities > 1)):\n            raise ValueError("Stop probabilities must lie in [0, 1]")\n        predictions = np.zeros_like(self.targets, dtype=np.int8)\n        seen = np.zeros(len(self.targets), dtype=np.int8)\n        thresholds = []\n        for fit, score in self.folds:\n            threshold = [_best_threshold(self.targets[fit, label], probabilities[fit, label])[0]\n                         for label in range(self.targets.shape[1])]\n            predictions[score] = probabilities[score] >= np.asarray(threshold)\n            seen[score] += 1\n            thresholds.append(threshold)\n        if not np.all(seen == 1):\n            raise RuntimeError("Every stop row must be scored once")\n        present = self.targets.sum(0) > 0\n        ap = float(average_precision_score(self.targets[:, present], probabilities[:, present], average="macro")) if present.any() else 0.0\n        return {"stop_macro_f1": float(f1_score(self.targets, predictions, average="macro", zero_division=0)),\n                "stop_macro_ap": ap, "stop_labels_with_positives": int(present.sum()),\n                "threshold_crossfit_splits": len(self.folds), "thresholds_by_fold": thresholds}\n\n\ndef better(score, best, tolerance=1e-6):\n    """F1 selects; AP is only a deterministic secondary criterion for F1 ties."""\n    if best is None:\n        return True\n    delta = score["stop_macro_f1"] - best["stop_macro_f1"]\n    return delta > tolerance or (abs(delta) <= tolerance and score["stop_macro_ap"] > best["stop_macro_ap"] + tolerance)\n', 'sparse.py': '"""Text-only NB log-count-ratio + logistic regression for nine symptoms.\n\nThis is NB-SVM-inspired feature reweighting with logistic regression, not an\nSVM. Vocabulary, IDF, count ratios and classifiers are fitted only from the\nexplicit training texts and targets. Prediction never updates those objects.\n"""\nfrom collections.abc import Mapping\nimport numbers\nimport unicodedata\n\nimport numpy as np\nfrom scipy import sparse\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\n\n\nN_LABELS = 9\nSCHEMA_VERSION = 1\nDEFAULT_CONFIG = {\n    "char_max_features": 80000,\n    "word_max_features": 20000,\n    "min_df": 2,\n    "C": 1.0,\n    "max_iter": 500,\n    "alpha": 1.0,\n    "class_weight": None,\n    "seed": 42,\n}\n\n\ndef _texts(values):\n    if isinstance(values, (str, bytes, Mapping)):\n        raise TypeError("texts must be an iterable of text strings, never annotations")\n    try:\n        values = list(values)\n    except TypeError as error:\n        raise TypeError("texts must be an iterable of text strings") from error\n    if any(not isinstance(value, str) for value in values):\n        raise TypeError("texts must contain text strings only, never annotations")\n    # The same label-independent normalization applies at fit and inference.\n    return [" ".join(unicodedata.normalize("NFC", value).split()) for value in values]\n\n\ndef _targets(values, n_samples):\n    values = np.asarray(values)\n    if values.shape != (n_samples, N_LABELS):\n        raise ValueError(f"targets must have shape ({n_samples}, {N_LABELS})")\n    if not (np.issubdtype(values.dtype, np.number) or values.dtype == np.bool_):\n        raise ValueError("targets must be numeric binary indicators")\n    if np.iscomplexobj(values) or not np.isfinite(values).all() or not np.isin(values, [0, 1]).all():\n        raise ValueError("targets must contain finite binary indicators 0 or 1")\n    return values.astype(np.int8, copy=False)\n\n\ndef _config(config):\n    if config is not None and not isinstance(config, Mapping):\n        raise TypeError("config must be a mapping")\n    supplied = dict(config or {})\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown sparse config keys: {sorted(unknown)}")\n    result = {**DEFAULT_CONFIG, **supplied}\n    for key in ("char_max_features", "word_max_features", "min_df", "max_iter"):\n        value = result[key]\n        if isinstance(value, bool) or not isinstance(value, numbers.Integral) or value < 1:\n            raise ValueError(f"{key} must be a positive integer")\n        result[key] = int(value)\n    if result["max_iter"] > 500:\n        raise ValueError("max_iter must not exceed 500")\n    for key in ("C", "alpha"):\n        value = result[key]\n        if isinstance(value, bool) or not isinstance(value, numbers.Real) or not np.isfinite(value) or value <= 0:\n            raise ValueError(f"{key} must be finite and positive")\n        result[key] = float(value)\n    if result["class_weight"] not in (None, "balanced"):\n        raise ValueError("class_weight must be None or \'balanced\'")\n    if isinstance(result["seed"], bool) or not isinstance(result["seed"], numbers.Integral):\n        raise ValueError("seed must be an integer")\n    result["seed"] = int(result["seed"])\n    if not 0 <= result["seed"] <= 2**32 - 1:\n        raise ValueError("seed must be between 0 and 2**32 - 1")\n    return result\n\n\ndef _fit_features(texts, config):\n    matrices, vectorizers = [], []\n    for analyzer, ngram_range, maximum in (\n        ("char", (2, 5), config["char_max_features"]),\n        ("word", (1, 2), config["word_max_features"]),\n    ):\n        vectorizer = TfidfVectorizer(\n            analyzer=analyzer, ngram_range=ngram_range, min_df=config["min_df"],\n            max_features=maximum, lowercase=False, sublinear_tf=True,\n            dtype=np.float32, token_pattern=r"(?u)\\b\\w+\\b",\n        )\n        try:\n            matrix = vectorizer.fit_transform(texts)\n        except ValueError as error:\n            # Tiny/all-blank inputs can have no qualifying n-grams in one or\n            # both channels. Keep the sample and use the remaining features.\n            expected = ("empty vocabulary", "After pruning, no terms remain", "max_df corresponds to < documents than min_df")\n            if not any(message in str(error) for message in expected):\n                raise\n            vectorizer = None\n        else:\n            matrices.append(matrix)\n        vectorizers.append(vectorizer)\n    matrix = (sparse.hstack(matrices, format="csr", dtype=np.float32) if matrices\n              else sparse.csr_matrix((len(texts), 0), dtype=np.float32))\n    return vectorizers, matrix\n\n\ndef fit_model(texts, targets, config=None):\n    """Fit a joblib-serializable artifact using only the supplied fit split."""\n    texts = _texts(texts)\n    if not texts:\n        raise ValueError("At least one training text is required")\n    targets = _targets(targets, len(texts))\n    config = _config(config)\n    vectorizers, features = _fit_features(texts, config)\n    estimators = []\n    for index in range(N_LABELS):\n        labels = targets[:, index]\n        positives = int(labels.sum())\n        if positives in (0, len(texts)) or features.shape[1] == 0:\n            estimators.append({"constant": float(positives / len(texts))})\n            continue\n        positive_mass = np.asarray(features[labels == 1].sum(axis=0), dtype=np.float64).ravel() + config["alpha"]\n        negative_mass = np.asarray(features[labels == 0].sum(axis=0), dtype=np.float64).ravel() + config["alpha"]\n        # Sum-normalized smoothed feature masses, as in NB log-count ratios.\n        ratio = (np.log(positive_mass / positive_mass.sum())\n                 - np.log(negative_mass / negative_mass.sum())).astype(np.float32)\n        if not np.isfinite(ratio).all():\n            raise ValueError("Non-finite NB feature ratios")\n        classifier = LogisticRegression(\n            C=config["C"], max_iter=config["max_iter"], class_weight=config["class_weight"],\n            solver="liblinear", n_jobs=1, random_state=config["seed"],\n        )\n        classifier.fit(features.multiply(ratio).tocsr(), labels)\n        estimators.append({"ratio": ratio, "classifier": classifier})\n    return {\n        "schema_version": SCHEMA_VERSION,\n        "model_type": "nb_log_count_ratio_logistic_regression",\n        "n_labels": N_LABELS,\n        "n_features": int(features.shape[1]),\n        "fit_samples": len(texts),\n        "config": config,\n        "vectorizers": vectorizers,\n        "estimators": estimators,\n    }\n\n\ndef _validate_model(model):\n    if not isinstance(model, Mapping) or model.get("schema_version") != SCHEMA_VERSION:\n        raise ValueError("Unsupported sparse model schema")\n    if model.get("model_type") != "nb_log_count_ratio_logistic_regression" or model.get("n_labels") != N_LABELS:\n        raise ValueError("Sparse model must be a nine-label NB logistic regression artifact")\n    count = model.get("n_features")\n    if isinstance(count, bool) or not isinstance(count, numbers.Integral) or count < 0:\n        raise ValueError("Invalid sparse feature count")\n    vectorizers, estimators = model.get("vectorizers"), model.get("estimators")\n    if not isinstance(vectorizers, (tuple, list)) or len(vectorizers) != 2:\n        raise ValueError("Sparse model requires two vectorizer slots")\n    vocabulary_size = 0\n    for vectorizer in vectorizers:\n        if vectorizer is None:\n            continue\n        if not isinstance(vectorizer, TfidfVectorizer) or not hasattr(vectorizer, "vocabulary_"):\n            raise ValueError("Invalid or unfitted sparse vectorizer")\n        if not np.isfinite(vectorizer.idf_).all():\n            raise ValueError("Non-finite vectorizer IDF")\n        vocabulary_size += len(vectorizer.vocabulary_)\n    if vocabulary_size != count:\n        raise ValueError("Vectorizer vocabulary and sparse feature count differ")\n    if not isinstance(estimators, (tuple, list)) or len(estimators) != N_LABELS:\n        raise ValueError("Sparse model requires nine label estimators")\n    for estimator in estimators:\n        if not isinstance(estimator, Mapping):\n            raise ValueError("Invalid sparse label estimator")\n        if "constant" in estimator:\n            value = estimator["constant"]\n            if not isinstance(value, numbers.Real) or not np.isfinite(value) or not 0 <= value <= 1:\n                raise ValueError("Constant probability must be finite and within [0, 1]")\n            continue\n        ratio = np.asarray(estimator.get("ratio"))\n        if ratio.shape != (count,) or not np.issubdtype(ratio.dtype, np.floating) or not np.isfinite(ratio).all():\n            raise ValueError("Invalid NB feature ratio shape or values")\n        classifier = estimator.get("classifier")\n        if (not isinstance(classifier, LogisticRegression) or not hasattr(classifier, "classes_")\n                or not np.array_equal(classifier.classes_, [0, 1])\n                or getattr(classifier, "n_features_in_", None) != count\n                or not np.isfinite(classifier.coef_).all()\n                or not np.isfinite(classifier.intercept_).all()):\n            raise ValueError("Invalid fitted binary logistic regression estimator")\n\n\ndef predict_model(model, texts):\n    """Return finite float32 probabilities of shape [n, 9] without fitting."""\n    _validate_model(model)\n    texts = _texts(texts)\n    if not texts:\n        return np.empty((0, N_LABELS), dtype=np.float32)\n    matrices = [vectorizer.transform(texts) for vectorizer in model["vectorizers"] if vectorizer is not None]\n    features = (sparse.hstack(matrices, format="csr", dtype=np.float32) if matrices\n                else sparse.csr_matrix((len(texts), 0), dtype=np.float32))\n    if features.shape[1] != model["n_features"] or not np.isfinite(features.data).all():\n        raise ValueError("Invalid inference feature shape or values")\n    output = np.empty((len(texts), N_LABELS), dtype=np.float32)\n    for index, estimator in enumerate(model["estimators"]):\n        if "constant" in estimator:\n            output[:, index] = estimator["constant"]\n        else:\n            output[:, index] = estimator["classifier"].predict_proba(\n                features.multiply(estimator["ratio"]).tocsr())[:, 1]\n    if not np.isfinite(output).all() or (output < 0).any() or (output > 1).any():\n        raise ValueError("Invalid sparse output probabilities")\n    return output\n', 'train.py': '"""Training-only worker; use run.py/the notebook for the shared hard deadline."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport torch\nif __package__:\n    from .pipeline import train_pipeline\nelse:\n    from pipeline import train_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("train_label_dir", "base_dir", "pretrained_dir", "output_dir"):\n        parser.add_argument("--" + name, required=True)\n    parser.add_argument("--config_json")\n    parser.add_argument("--start_epoch", type=float, required=True)\n    parser.add_argument("--deadline_epoch", type=float, required=True)\n    parser.add_argument("--auto_batch", action="store_true")\n    parser.add_argument("--attempt_marker", required=True, help="Local receipt for this run\'s verified portable model")\n    args = parser.parse_args()\n    config = json.loads(Path(args.config_json).read_text(encoding="utf-8")) if args.config_json else {}\n    if args.auto_batch:\n        identity = Path(args.output_dir) / "run_identity.json"\n        if identity.is_file():\n            saved = json.loads(identity.read_text())["config"]\n            for key in ("batch_size", "window_batch_size", "gradient_accumulation_steps", "inference_batch_size"):\n                config.setdefault(key, saved[key])\n        elif torch.cuda.is_available():\n            memory = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3\n            batch = 8 if memory >= 35 else 4 if memory >= 22 else 2\n            config.setdefault("batch_size", batch)\n            config.setdefault("window_batch_size", batch)\n            config.setdefault("gradient_accumulation_steps", 32 // batch)\n            config.setdefault("inference_batch_size", batch)\n    report = train_pipeline(args.train_label_dir, args.base_dir, args.output_dir, config, args.pretrained_dir,\n                            args.start_epoch, args.deadline_epoch, args.attempt_marker)\n    print("Training-only selection:", report["selected_name"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': '"""Competition interface: text-only inference, then optional read-only scoring."""\nimport argparse\nimport json\nfrom pathlib import Path\nif __package__:\n    from .pipeline import predict_pipeline, evaluate_csv, write_json\nelse:\n    from pipeline import predict_pipeline, evaluate_csv, write_json\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--audio_dir", default="", help="Accepted for the required interface; audio is not read")\n    for name in ("label_dir", "ckpt_path", "output"):\n        parser.add_argument("--" + name, required=True)\n    parser.add_argument("--device", choices=("cpu", "cuda"))\n    parser.add_argument("--batch_size", type=int, default=4)\n    parser.add_argument("--cache_dir")\n    parser.add_argument("--candidate_output")\n    parser.add_argument("--evaluate", action="store_true")\n    args = parser.parse_args()\n    paths, _, _ = predict_pipeline(args.ckpt_path, args.label_dir, args.output, args.device,\n                                   args.batch_size, args.cache_dir, args.candidate_output)\n    if args.evaluate:\n        if all("symptom" in json.loads(p.read_text(encoding="utf-8-sig")) for p in paths):\n            for csv_path, report_name in ((args.output, "validation_metrics.json"),\n                                           (args.candidate_output, "candidate_validation_metrics.json")):\n                if csv_path and Path(csv_path).is_file():\n                    report = evaluate_csv(args.label_dir, csv_path)\n                    write_json(Path(csv_path).with_name(report_name), report)\n                    print(f"{report_name}: Macro F1 = {report[\'macro_f1\']:.6f}", flush=True)\n        else:\n            print("CSV saved; some labels have no targets, so scoring is skipped.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'run.py': '"""Standard-library supervisor for a single <= 3 hour automatic Colab run."""\nfrom __future__ import annotations\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\nimport time\nif __package__:\n    from .budget import TimeBudget, BudgetExceeded\nelse:\n    from budget import TimeBudget, BudgetExceeded\n\nHERE = Path(__file__).resolve().parent\nLAST_REPORT = {}\n\n\ndef _write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef run(settings, start_epoch=None):\n    global LAST_REPORT\n    budget = TimeBudget(start_epoch=start_epoch)\n    stages, failures = {}, []\n    # Do not resolve/stat Drive paths in the unsupervised notebook process.\n    output = Path(os.path.abspath(os.path.expanduser(settings["output_dir"])))\n    final_validation_completed = False\n    config = dict(settings.get("train_config", {}))\n    reserve = float(config.get("final_reserve_seconds", 1500))\n    if not 1500 <= reserve <= 3600:\n        raise ValueError("final_reserve_seconds must be between 1500 and 3600")\n    common_env = dict(os.environ, HF_HUB_DISABLE_TELEMETRY="1", TOKENIZERS_PARALLELISM="false",\n                      OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")\n\n    def call(stage, arguments, *, online=False, reserve_seconds=0, timeout=None):\n        before = time.time()\n        print(f"\\n[{stage}] remaining {budget.remaining_seconds() / 60:.1f} minutes", flush=True)\n        offline = "0" if online else "1"\n        try:\n            return budget.run([sys.executable, "-u", *map(str, arguments)], cwd=HERE,\n                reserve_seconds=reserve_seconds, timeout=timeout, stream_output=True,\n                env=dict(common_env, HF_HUB_OFFLINE=offline, TRANSFORMERS_OFFLINE=offline))\n        finally:\n            stages[stage] = time.time() - before\n\n    try:\n        if settings.get("install_dependencies", True):\n            call("dependencies", ["-m", "pip", "install", "--disable-pip-version-check", "-q", "-r", HERE / "requirements.txt"],\n                 online=True, reserve_seconds=reserve + 2400, timeout=900)\n        if settings.get("input_zip"):\n            destination = Path(os.path.abspath(os.path.expanduser(settings.get("extract_dir", "/content/mission3_ver5_inputs"))))\n            # Restore in a supervised process; large Drive I/O cannot block the notebook indefinitely.\n            script = "import sys; from bootstrap import restore_input_bundle; restore_input_bundle(sys.argv[1], sys.argv[2])"\n            call("input_restore", ["-c", script, settings["input_zip"], destination], reserve_seconds=reserve + 2400)\n            bundle = destination / "DCC_AIdeas"\n            base = bundle / "results/mission3/ver4"\n            data_root = bundle / "DATA/대학부 데이터"\n            training = data_root / "Training/2.라벨링데이터/TL_서울_구급"\n            validation = data_root / "Validation/2.라벨링데이터/VL_서울_구급"\n        else:\n            base = Path(os.path.abspath(settings["base_dir"]))\n            training = Path(os.path.abspath(settings["train_label_dir"]))\n            validation = Path(os.path.abspath(settings["valid_label_dir"])) if settings.get("valid_label_dir") else None\n        public_model = Path(os.path.abspath(os.path.expanduser(settings.get("pretrained_dir", "/content/mission3_ver9_pretrained/deberta_korean"))))\n        call("public_pretrained", [HERE / "pretrained.py", "--destination", public_model],\n             online=True, reserve_seconds=reserve + 2400, timeout=1800)\n        config_path = HERE / "run_train_config.json"\n        _write(config_path, config)\n        attempt_marker = Path(tempfile.mkdtemp(prefix="mission3_ver9_attempt_")) / "portable_model_ready.json"\n        try:\n            call("training", [HERE / "train.py", "--train_label_dir", training, "--base_dir", base,\n                 "--pretrained_dir", public_model, "--output_dir", output, "--config_json", config_path,\n                 "--start_epoch", budget.start_epoch, "--deadline_epoch", budget.deadline_epoch, "--auto_batch",\n                 "--attempt_marker", attempt_marker],\n                 reserve_seconds=reserve)\n        except (BudgetExceeded, subprocess.CalledProcessError) as error:\n            failures.append({"phase": "training", "error": str(error)})\n            print("Candidate training interrupted. Checking the already published fallback.", flush=True)\n        if not attempt_marker.is_file():\n            raise RuntimeError("This attempt did not verify a portable model. Inspect training errors; an older run will not be evaluated as the new experiment.")\n        checkpoint = output / "mission3_ver9_model.ckpt"\n        if settings.get("run_validation", True):\n            if validation is None:\n                raise FileNotFoundError("Final Validation JSON directory is missing")\n            cache = Path(settings.get("cache_dir", output.parent / "prediction_cache"))\n            arguments = [HERE / "inference.py", "--audio_dir", "", "--label_dir", validation,\n                         "--ckpt_path", checkpoint, "--output", output / "inference.csv", "--cache_dir", cache,\n                         "--batch_size", settings.get("inference_batch_size", 4), "--evaluate"]\n            # Publish the requested final CSV first. Diagnostics cannot endanger it.\n            call("final_validation", arguments, reserve_seconds=30)\n            final_validation_completed = True\n            if settings.get("run_candidate_diagnostic", True) and budget.remaining_seconds() > 360:\n                try:\n                    call("candidate_diagnostic", [*arguments, "--candidate_output", output / "candidate_inference.csv"], reserve_seconds=20)\n                except (BudgetExceeded, subprocess.CalledProcessError) as error:\n                    failures.append({"phase": "candidate_diagnostic", "error": str(error)})\n        else:\n            call("verify_model", ["-c", "import sys; from pipeline import load_manifest; load_manifest(sys.argv[1])", checkpoint], reserve_seconds=20)\n        return output\n    except (BudgetExceeded, subprocess.CalledProcessError) as error:\n        failures.append({"phase": "workflow", "error": str(error)})\n        print("Automatic run stopped within the shared budget; completed artifacts are preserved.", flush=True)\n        raise\n    finally:\n        # Even a small Drive write may hang: keep a local report, copy under the same watchdog.\n        report = {"start_epoch": budget.start_epoch, "deadline_epoch": budget.deadline_epoch,\n            "elapsed_seconds": time.time() - budget.start_epoch, "budget_seconds": 10800,\n            "stages_seconds": stages, "failures": failures,\n            "final_validation_completed_this_run": final_validation_completed}\n        local_status = Path(tempfile.mkdtemp(prefix="mission3_ver9_status_")) / "runtime.json"\n        _write(local_status, report)\n        LAST_REPORT = {"runtime.json": report}\n        status_script = \'\'\'import json,sys,time\nfrom pathlib import Path\np=Path(sys.argv[2]); p.parent.mkdir(parents=True,exist_ok=True)\nv=json.loads(Path(sys.argv[1]).read_text())\nv[\'final_csv_exists\']=(p.parent/\'inference.csv\').is_file()\nv[\'elapsed_seconds\']=time.time()-v[\'start_epoch\']\nt=p.with_name(p.name+\'.tmp\')\nt.write_text(json.dumps(v,ensure_ascii=False,indent=2)+\'\\\\n\',encoding=\'utf-8\'); t.replace(p)\nsummary={\'runtime.json\':v}\nfor name in (\'training_metrics.json\',\'validation_metrics.json\',\'candidate_validation_metrics.json\'):\n    f=p.parent/name\n    if f.is_file(): summary[name]=json.loads(f.read_text(encoding=\'utf-8\'))\nprint(json.dumps(summary,ensure_ascii=False))\n\'\'\'\n        try:\n            saved = budget.run([sys.executable, "-c", status_script, str(local_status), str(output / "runtime.json")],\n                               timeout=10, capture_output=True, text=True)\n            LAST_REPORT = json.loads(saved.stdout)\n        except (BudgetExceeded, subprocess.CalledProcessError, ValueError):\n            print("Drive status save did not finish before the deadline. Local report:", local_status, flush=True)\n        print("Runtime report:", json.dumps(report, ensure_ascii=False), flush=True)\n        print("Output:", output, flush=True)\n        print(f"Total elapsed: {(time.time() - budget.start_epoch) / 60:.1f} minutes", flush=True)\n        if failures:\n            print("Interrupted phases:", json.dumps(failures, ensure_ascii=False), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--settings", required=True)\n    parser.add_argument("--start_epoch", type=float)\n    args = parser.parse_args()\n    run(json.loads(Path(args.settings).read_text(encoding="utf-8")), args.start_epoch)\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\nsentencepiece>=0.2,<0.3\nprotobuf>=4.25,<7\n', 'README.md': '# Mission 3 ver9 — Korean DeBERTa-v3 + NB-log-count LR\n\n`../mission3_ver9.ipynb` 한 개를 Colab에 올리고, ver8 때 쓰던 `mission3_ver5_inputs.zip`을 그대로 사용합니다.\n**아직 전체 GPU 학습과 ver9 F1은 측정하지 않았습니다.** 새 실행 결과와 기존 ver4 비교를 저장하도록 구현했습니다.\n\n## 실행\n\n1. 공유 ZIP에 **정리 → 바로가기 추가 → 내 드라이브**를 적용합니다. 기본 위치는 `/content/drive/MyDrive/mission3_ver5_inputs.zip`입니다.\n2. `mission3_ver9.ipynb`를 Colab에서 열고 GPU 런타임을 선택합니다. L4를 우선 사용하고 T4에서는 자동으로 작은 microbatch를 사용합니다.\n3. **모두 실행**합니다. Drive를 연결한 계정에 ZIP 바로가기가 있어야 합니다.\n4. 결과는 `내 드라이브/DCC_AIdeas/results/mission3/ver9`에 저장됩니다. 새 설정으로 실험하려면 `OUTPUT_DIR`을 새 폴더로 바꾸세요.\n\n추가 소스 업로드는 필요 없습니다. 실행 코드와 requirements가 노트북에 포함됩니다. 공개 모델만 자동 다운로드하고,\nTraining·Validation 자료를 외부 API로 보내지 않습니다. 모든 작업은 노트북의 한 절대 마감을 공유합니다.\n설정 셀부터 설치, ZIP 복원, 공개 가중치 준비, 학습, 보정, 최종 추론·평가까지 **최대 10,800초 예산**입니다.\nDrive 연결 대기나 수동 셀 간 대기 역시 설정 셀 이후면 예산에서 차감됩니다.\n\n최종 추론을 위해 최소 25분, Training 보정을 위해 추가 10분을 남깁니다. 첫 fold의 실제 소요 시간을 보고\n두 번째 fold를 실행할지 결정하고, epoch마다 시간 부족 여부를 확인합니다. 외부 작업은 제한 시간이 지나면 자식 프로세스까지 종료합니다.\n다운로드·Drive 장애나 GPU 속도 때문에 **3시간 안에 모든 학습과 평가의 완료를 보장하지는 않습니다.**\n후보 실행이 늦어지면 먼저 완성해 둔 기존 모델 묶음을 사용하며, 추론 자체가 끝나지 않으면 완료된 모델·기록을 보존합니다.\n자동으로 새 3시간 실행을 이어 붙이지 않습니다. 새 Colab 세션에서 사용자가 다시 실행하면 별도 3시간 예산이며 누적 실행 비용은 추가됩니다.\n\n## 변경 내용\n\n- `team-lucid/deberta-v3-base-korean` 공개 encoder 전체 미세조정. 고정 revision은 `47a3b00828c5dc2b8b080c6738ef7a58ebd419d5`입니다.\n- CLS + mean pooling, 증상별 window attention; 모든 대화의 512-token window와 64-token overlap을 유지합니다.\n- unweighted BCE, encoder LR `2e-5`, head LR `1e-4`, 최대 3 epoch, patience 2. 유효 batch는 32입니다.\n- 최대 2개 Training fold의 학습된 최적 모델 확률을 평균합니다. 두 번째 fold 실행 여부는 시간으로만 정합니다.\n- 문자·단어 TF-IDF에 증상별 NB log-count-ratio를 곱한 Logistic Regression을 추가합니다. SVM 구현이 아닙니다.\n- 새 후보는 `0.75 × DeBERTa 평균 + 0.25 × NB-LR`로 미리 고정합니다.\n- 후보와 기존 ver4 혼합은 Training calibration의 grouped cross-fit에서 `F1 +0.002` 및 최소 3개 fold 개선을 모두 만족할 때 채택합니다.\n- 조건을 통과하지 못하면 기존 ver4의 원래 가중치·보정값을 그대로 사용합니다. 후보 단독 결과는 별도로 기록합니다.\n\n자세한 기존 결과 분석, 논문, 해외 원문 자료와 규칙 근거는 [RESEARCH.md](RESEARCH.md)에 있습니다.\n공식 Validation은 선택 완료 뒤 평가에만 쓰며 학습 함수는 Validation 경로를 받지 않습니다.\nNB 모델도 공통 Training calibration을 제외하고 학습합니다. 같은 전사의 그룹은 fit/stop/cal 사이에 섞이지 않게 검증합니다.\n\n## 결과 파일\n\n| 파일 | 내용 |\n| --- | --- |\n| `mission3_ver9_model.ckpt` | 최종 조합·임계값·상대 경로·SHA256을 담은 JSON manifest |\n| `base/` | 기존 ver4의 실제 5개 `.ckpt`, sparse 모델, tokenizer, manifest |\n| `candidate_1.ckpt`, 선택적 `candidate_2.ckpt` | 새 DeBERTa의 실제 최적 가중치 |\n| `candidate_*_last.ckpt` | 중간 재개용 optimizer 포함 상태; 완료된 모델은 최종 선택 저장 후 정리 |\n| `nb_ratio_lr.joblib`, `tokenizer/` | 새 희소 분류기와 DeBERTa tokenizer |\n| `run_identity.json`, `candidate_history.json` | 데이터·코드·모델 신원 및 epoch 기록 |\n| `training_metrics.json`, `calibration_comparison.json` | Training 내부 선택과 채택/미채택 이유 |\n| `inference.csv`, `validation_metrics.json` | 최종 예측 및 최종 Macro F1 |\n| `candidate_inference.csv`, `candidate_validation_metrics.json` | 시간이 남을 때만 수행하는 후보 단독 진단 |\n| `runtime.json` | 설치부터 마지막 단계까지 실제 시간과 중단 사유 |\n| `submission/` | 독립 실행 코드, `model.py`, `inference.py`, requirements 사본 |\n\n결과 폴더는 원래 ver4 형제 폴더 없이 이동할 수 있도록 실제 base 파일을 복사합니다. 따라서 **최소 약 10~15 GB의 추가 Drive 공간**을 권장합니다.\n기존 입력 ZIP 용량은 별도입니다. 정확한 필요 공간은 후보 모델 수와 중간 optimizer 저장 상태에 따라 달라집니다.\n작은 `.ckpt` manifest 하나만 내려받으면 추론할 수 없습니다. 결과 폴더 전체가 필요합니다.\n\n아래 명령은 문제의 네 인자를 그대로 받으며 경로를 옮겨도 별도 `--base_dir`가 필요 없습니다.\n\n```bash\npip install -r results/mission3/ver9/submission/requirements.txt\npython results/mission3/ver9/submission/inference.py \\\n  --audio_dir "DATA/대학부 데이터/Validation/1.원천데이터/VS_서울_구급" \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --ckpt_path results/mission3/ver9/mission3_ver9_model.ckpt \\\n  --output results/mission3/ver9/inference.csv\n```\n\n`--audio_dir`는 규정 인터페이스 호환용이며 음성을 읽지 않습니다. 입력은 양측 전사 text뿐입니다.\nCSV 헤더는 `label file name,symptom`이며 증상은 지정 9개 중 0~9개의 문자열 목록입니다.\n학습/추론은 준비된 로컬 가중치만 읽습니다. 공개 모델 다운로드는 별도 준비 단계에서만 수행합니다.\n\n## 재개와 검증 범위\n\n학습은 완료된 epoch 단위로 재개합니다. 데이터·설정·코드·공개 revision·GPU/정밀도가 달라지면 무조건 이어서 학습하지 않습니다.\n중간 epoch는 최적 가중치로 발표하지 않습니다. 완료된 실험은 최종 선택을 재사용하며 Validation 결과로 재선택하지 않습니다.\n새로운 실험 설정에는 새 결과 폴더를 사용하세요.\n\nCPU의 작은 DeBERTa와 합성 데이터로 전체 encoder 업데이트, 긴 대화 꼬리 gradient, 정확한 epoch 재개,\n부분 epoch 중단, 순수 추론, 희소 분류기, 그룹 보정, 시간 초과 자식 종료와 제출 CSV 흐름을 검증합니다.\n이는 실제 한국어 pretrained 가중치의 Colab GPU 속도나 F1 검증을 대신하지 않습니다.\n\n```bash\nPYTHONPATH=src/mission3 python -m pytest src/mission3/mission3_ver9/tests -q\npython src/mission3/mission3_ver9/build_notebook.py\n```\n\n제출 전에는 Colab 실행 결과를 저장한 학습 `.ipynb`, 실제 가중치·추론 코드, requirements, 결과 PPT를 준비해야 합니다.\n현재 저장소의 새 노트북은 실행 전 템플릿이며, 측정 점수·학습 출력·PPT를 임의로 만들어 넣지 않았습니다.\n', 'RESEARCH.md': '# Mission 3 ver9 설계 근거\n\n작성일: 2026-09-17. **ver9의 전체 데이터 GPU 학습, Validation F1, 실제 Colab 소요 시간은 아직 측정하지 않았습니다.**\n이 문서는 기존 산출물에서 확인된 사실과 새 실험의 가설을 구분합니다. 성능 개선을 보장하는 문서가 아닙니다.\n\n## 1. ver1~ver8에서 확인된 결과\n\n근거는 [experiment_log.md](../experiment_log.md)의 버전별 기록과 `results/mission3/`의 저장된 평가입니다.\n표의 Validation은 서울 3,640개 전체에 대한 Macro F1입니다.\n\n| 버전 | 실험 | 최종 Validation Macro F1 | 해석 |\n| --- | --- | --- | --- |\n| ver1 | 모듈형 TF-IDF + Logistic Regression | 별도 실행 기록 없음 | 코드 구현과 실제 측정 구분 |\n| ver2 | 단일 노트북 TF-IDF + Logistic Regression | 약 0.6001 | 최초 저장 baseline; 결과 폴더 이름은 `ver1` |\n| ver3 | KLUE-RoBERTa-base 미세조정 | 약 0.6260 | 전통적 TF-IDF baseline보다 개선 |\n| ver4 | KLUE-RoBERTa-large 5-fold + TF-IDF/LR | 0.656648948 | 현재 주력 앙상블 |\n| ver5 | 증상 attention 추가 학습·혼합 보정 | 0.656717758 | ver4 대비 약 +0.0069%p; 최적 후보는 추가 학습 전 epoch 0 |\n| ver6 | A–F 분류기/손실 비교·D 미세조정 | 0.656669718 | 새 후보 미채택, 기존 ver4 구성 사용 |\n| ver7 | KoELECTRA-Base-v3 전체 미세조정 | 0.656669718 | 새 후보 미채택; 후보 단독은 0.638109662 |\n| ver8 | RoBERTa-large + 증상별 token attention 전체 미세조정 | 0.656669718 | 새 후보 미채택; 후보 단독은 0.645292165 |\n\nver6~ver8의 최종 CSV는 동일한 예측입니다. 따라서 세 버전의 최종 점수를 각 새 encoder의 성능으로 설명하면 안 됩니다.\nver5의 저장된 최종 점수가 수치상 가장 높지만, 이 정도 차이만으로 추가 학습 효과나 일반화 개선을 확정할 수 없습니다.\n\n### 실패 원인으로 단정할 수 있는 것과 없는 것\n\n- **채택 기준만 낮추면 해결된다는 증거는 없습니다.** ver8의 최선 혼합 calibration 교차평가 F1은\n  0.657450867로 기존 0.659670866보다 낮았습니다. `+0.002` 기준을 제거해도 이번 후보의 혼합 성능은 기존보다 낮습니다.\n- ver6의 최선 logit 혼합은 0.662117로 기존보다 높았지만 개선 fold가 2/5여서 일관성 기준을 통과하지 못했습니다.\n  특정 fold에서의 개선을 모든 통화로 일반화하지 않도록 이 기준을 유지합니다.\n- 새 모델 1개와 ver4의 5개 모델 앙상블은 학습량·분산 감소 효과가 다릅니다. 다만 동일한 stop에서의 비교도\n  ver8 후보 0.637975045가 ver4 fold 1 단독 0.639684092보다 낮았습니다. 단순히 앙상블 개수 차이만으로 결과를 설명할 수도 없습니다.\n- ver6은 ASL, R-Drop 등을 head 수준에서 이미 비교했습니다. 같은 손실을 이름만 바꾸어 새 개선책으로 제시하지 않습니다.\n  [Asymmetric Loss 논문](https://arxiv.org/abs/2009.14119)과 [R-Drop 논문](https://arxiv.org/abs/2106.14448)의\n  다른 과제 성과는 이 데이터에서의 개선 보장이 아닙니다. 전체 encoder에 R-Drop을 적용하면 추가 forward 비용도 듭니다.\n- 전사와 symptom의 대응이 불명확해 보이는 사례만으로 라벨 오류율이나 F1의 상한을 추정할 수 없습니다.\n  이번 실험은 정답 자동 수정·외부 LLM 재라벨링·특정 샘플 제거를 하지 않습니다.\n- ver8의 저장된 CSV 점수는 검증되어 있지만 로컬 `trials/B/best.ckpt`는 완료 기록의 epoch 3과 해시가 다릅니다.\n  따라서 ver9는 이 불일치 후보를 초기 가중치로 사용하지 않습니다. 공개 고정 모델과 기존 ver4 입력 묶음에서 시작합니다.\n\n## 2. 새 encoder: 한국어 DeBERTa-v3-base\n\n사용 모델은 [team-lucid/deberta-v3-base-korean](https://huggingface.co/team-lucid/deberta-v3-base-korean)입니다.\n공개 encoder와 fast tokenizer를 다음 [고정 revision](https://huggingface.co/team-lucid/deberta-v3-base-korean/tree/47a3b00828c5dc2b8b080c6738ef7a58ebd419d5)에서 준비합니다.\n\n```text\nmodel_id: team-lucid/deberta-v3-base-korean\nrevision: 47a3b00828c5dc2b8b080c6738ef7a58ebd419d5\n```\n\n고정 revision의 실제 config/tokenizer를 Transformers 4.57.1에서 읽어 fast tokenizer 생성,\n긴 한국어 전사의 overflow window·마지막 문구 보존, 빈 입력, 저장 후 오프라인 재로딩을 확인했습니다.\n이는 전체 pretrained 가중치 다운로드나 GPU 학습 검증을 대신하지 않습니다.\nconfig의 최대 위치는 512이고 tokenizer의 `model_max_length`는 실제 제한을 뜻하지 않는 큰 sentinel 값이므로\n코드에서 길이 512를 명시합니다. 토큰화 API는 [Transformers tokenizer 문서](https://huggingface.co/docs/transformers/main_classes/tokenizer)를,\n관련 config는 [DeBERTa-v2 문서](https://huggingface.co/docs/transformers/model_doc/deberta-v2)를 참고하며,\n최신 온라인 문서와 실제 실행에 고정한 4.57.1 버전은 구분합니다.\n\n[DeBERTaV3 원논문](https://arxiv.org/abs/2111.09543)은 replaced-token detection과\ngradient-disentangled embedding sharing을 사용하는 사전학습을 설명합니다. 이 구조의 공개 한국어 모델로\nencoder를 교체하여 기존 RoBERTa와 다른 표현·오류 패턴을 얻을 수 있는지 확인합니다.\n모델 카드의 한국어 benchmark 결과는 모델 후보를 고르는 근거이며, 한국어 응급 통화 9-label F1에 대한 직접 증거는 아닙니다.\n\n이번 분류 학습은 공개 encoder 전체와 새 분류기를 함께 학습하는 **최대 3 epoch 미세조정**입니다.\nver4의 CLS + mean pooling / window attention 분류 구조를 사용하여 새 token attention까지 동시에 탐색하는 비용을 줄입니다.\n입력은 전체 양측 전사이며, 발화 사이에 동일한 `[SEP]`를 넣고 **512-token window / 64-token overlap**으로 처리합니다.\n발화자 정답이나 증상에 따라 window·전처리를 바꾸지 않습니다.\n\n손실은 기본 **unweighted BCE**입니다. 기존 양성 가중치가 적용된 BCE와 다른 정밀도/재현율 균형을 탐색하고,\n증상별 판정 임계값은 별도 Training calibration에서 정합니다. 양성 가중치가 기존 실패의 원인이라고 확정한 것은 아닙니다.\n여러 요소가 동시에 바뀌므로 이번 결과 하나로 encoder, 손실, pooling 각각의 인과적 기여를 분리할 수도 없습니다.\n\n기본적으로 첫 번째 fold를 학습합니다. 첫 fold에서 측정한 시간과 남은 전체 예산이 충분할 때만 두 번째 fold를 시작하고,\n완료·검증된 neural 모델들의 확률을 평균합니다. 이는 새 5-fold 앙상블 또는 다중 seed 확증 실험과 다릅니다.\n두 번째 fold 실행 여부는 시간으로 판단하며 공식 Validation 점수로 결정하지 않습니다.\n\n다른 후보인 [KcELECTRA](https://huggingface.co/beomi/KcELECTRA-base)는 댓글과 응급 통화의 도메인이 다르고,\n이번에는 ver7과 다른 encoder 구조를 우선하기 때문에 기본 후보로 채택하지 않았습니다.\n[mmBERT](https://huggingface.co/jhu-clsp/mmBERT-base)는\n[원논문](https://arxiv.org/abs/2509.06888)의 다국어·긴 문맥 설계가 있지만, 현재 Colab 제한에서 학습 속도와\n가속 호환성을 검증하지 않았고 한국어 응급 통화에서의 우위를 확인한 결과도 없어 기본 실행에서 제외했습니다.\n\n## 3. 독립적인 희소 모델: NB log-count-ratio + Logistic Regression\n\n[Wang & Manning, ACL 2012](https://aclanthology.org/P12-2018/)의 NB log-count-ratio 특징 재가중 아이디어를 사용합니다.\n[저자 공개 자료](https://nlp.stanford.edu/~sidaw/home/projects%3Anbsvm)도 참고했습니다.\n**이 구현의 분류기는 Logistic Regression이며 SVM이 아닙니다.** TF-IDF 질량을 사용하므로 원문의 NB-SVM을 그대로 재현한 구현이라고 부르지 않습니다.\n\n문자 2~5-gram TF-IDF 최대 80,000개와 단어 1~2-gram TF-IDF 최대 20,000개를 사용합니다.\n기본 `min_df=2`, `sublinear_tf=True`이며 각 채널의 TF-IDF를 이어 붙입니다.\n9개 증상마다 양성/음성 문서의 특징 질량에 1을 더해 평활화하고 각각 합으로 정규화한 뒤 로그 비율을 계산합니다.\n이 비율을 희소 특징에 곱한 결과로 `C=1`, `class_weight=None`, `max_iter=500`, `n_jobs=1`의 이진 Logistic Regression을 학습합니다.\n\n목표는 기존 TF-IDF/LR과 다른 증상별 어휘 가중치를 제공하는 것입니다. NB 비율이 항상 일반화를 개선하는 것은 아닙니다.\n희귀 특징의 과적합을 줄이기 위한 기본 빈도 제한·평활화는 있지만, 실제 기여는 Training 내부 교차평가와 최종 Validation으로 확인해야 합니다.\n\n이 희소 모델은 **Training calibration을 제외한 Training 전체**에서 vocabulary, IDF, NB 비율, LR을 학습합니다.\n일부 텍스트는 neural 모델의 stop에 해당할 수 있지만, 이 NB 모델로 neural epoch를 고르지 않습니다.\n모든 최종 후보 모델에서 공통 calibration 전체를 제외한다는 경계는 유지합니다.\n추론에서는 `transform`과 `predict_proba`만 호출하며 unseen 입력으로 vocabulary·IDF·분류기를 재학습하지 않습니다.\n한 증상에 양성 또는 음성이 전혀 없거나 추출할 특징이 없으면 해당 fit 분할에서 계산한 constant 확률을 사용합니다.\n\n## 4. 혼합·선택·평가 경계\n\n새 후보 확률은 다음 비율로 **미리 고정**합니다. DeBERTa와 NB 비율 자체를 calibration이나 Validation에서 다시 탐색하지 않습니다.\n\n```text\ncandidate = 0.75 × completed_DeBERTa_probability_mean + 0.25 × NB_logistic_probability\n```\n\n- neural 가중치 학습: 각 fold의 fit 데이터만 사용합니다.\n- neural epoch 선택: 해당 fold의 stop에서 그룹 분리 threshold 교차평가 Macro F1을 사용합니다. AP는 동점 보조 지표입니다.\n- NB 모델 학습: 서울 Training에서 공통 calibration을 제외한 데이터만 사용합니다.\n- 최종 보정: 기존 ver4 neural 확률, 기존 ver4 sparse 확률, 위의 고정된 새 후보를 공통 calibration에서 비교합니다.\n  calibration 그룹별 교차평가의 학습 쪽에서만 alpha·beta·threshold를 맞추고 제외된 그룹에서 점수를 계산합니다.\n  동일 전사 그룹이 보정 fit/score 양쪽으로 나뉘지 않도록 합니다.\n- 비교하는 혼합 계열은 raw 확률 평균과 logit 평균입니다. 증상별 가중치 탐색은 각 fold 안에서 수행하고,\n  두 계열 중 하나를 전체 교차평가 Macro F1로 선택합니다. 계열 점수가 같으면 raw를 선택하고 동점 가중치는 후보를 덜 쓰는 쪽을 선호합니다.\n- 채택은 기존보다 **Macro F1이 0.002 이상 증가**하고 **최소 3개 calibration fold에서 개선**될 때만 합니다.\n  실제 fold 수가 적어도 요구 개선 fold 수를 자동으로 낮추지 않습니다. 조건을 통과하지 못하면 후보 비율 0의 기존 ver4 구성을 유지합니다.\n- 후보 단독 진단의 임계값도 Training calibration에서 고정합니다.\n- 공식 Validation은 모든 가중치·혼합·임계값을 확정한 뒤 최종 CSV와 점수를 만드는 데만 사용합니다.\n  Validation 정답을 학습·epoch 선택·혼합 비율·threshold 탐색·예측 캐시 생성에 사용하지 않습니다.\n\n교차평가 점수로 혼합 계열을 다시 선택하므로 선택 편향은 남습니다. `+0.002 / 3 fold` 조건은 보수적 선택 규칙이며\n통계적 유의성 검정이나 독립 Test 성능 보장은 아닙니다. 여러 버전의 결과를 보고 다음 설계를 정한 연구 과정 자체도\n완전히 새로운 평가 데이터에 대한 확증 실험과 구분해야 합니다.\n\n## 5. 3시간 실행 예산\n\nver8은 3시간을 목표로 추정했지만 자동 중단이 없었습니다. ver9는 **한 번의 자동 실행 전체에 공유하는 10,800초 절대 마감**을 사용합니다.\n자동 준비가 시작될 때 만든 deadline을 패키지 설치·입력 복원·모델 준비·학습·최종 추론 단계로 전달하고,\n단계마다 새 3시간을 부여하지 않습니다. 최종 보정·저장·평가 시간을 먼저 남겨두고 추가 fold 또는 epoch의 실행 여부를 판단합니다.\n외부 Python 프로세스는 남은 시간으로 제한하며 마감에 도달하면 프로세스 그룹을 종료합니다.\n\n이 제한은 **3시간 안에 2-fold × 3 epoch와 모든 Validation 진단이 반드시 끝난다는 보장이 아닙니다.**\nGPU 종류, 통화별 window 수, 다운로드, Drive I/O에 따라 학습 fold/epoch 수 또는 완료 가능한 평가 범위가 달라집니다.\n시간이 부족하면 이미 완성된 가중치·기록을 보존하고 추가 작업을 멈춥니다. Colab 런타임 강제 종료, 사용자 Drive 인증 대기,\n시스템 자체의 정지와 같은 외부 상황까지 Python 코드가 통제할 수는 없습니다.\n\n기존 로그에서 ver7의 실제 epoch 학습 시간 합계는 약 37.30분, ver8은 약 63.06분이었습니다.\n이 값은 해당 실행의 단계 기록이고 설치·재개 이전 전체 시간을 포함하는 새 실행의 총비용 추정치가 아닙니다.\nDeBERTa가 더 작은 encoder라는 이유만으로 실제 wall-clock 시간이나 F1 향상을 단정하지 않습니다.\n최종 판단은 ver9에 저장된 실행 시간·완료 상태·CSV·평가 JSON으로 합니다.\n\n## 6. 문제 PDF와의 대응\n\n기준은 [0820_DCC 대학부_예선_출제문제.pdf](../../../docs/0820_DCC%20대학부_예선_출제문제.pdf)입니다.\n\n| PDF 페이지 | 조건 | ver9 적용 |\n| --- | --- | --- |\n| 5 | AI-hub 제공 데이터만 사용, 외부 데이터 추가 금지 | 서울 Training 전사와 정답만 학습에 사용; 외부 학습 표본·LLM 생성 표본 없음 |\n| 5 | 공개 사전학습 모델 허용, 별도의 개발 없는 상용 AI API 금지 | 고정 revision 공개 DeBERTa를 로컬에서 미세조정; API 추론·데이터 전송 없음 |\n| 5 | Training은 서울만, Validation 학습 금지 | fit/stop/calibration 모두 서울 Training 내부; 공식 Validation은 최종 평가만 |\n| 5 | 라벨별 차등 전처리, 평가 데이터만 유리한 전처리 금지 | 모든 표본에 동일한 text 정규화·발화 구분·window 규칙 적용 |\n| 9 | 입력은 양측 대화의 전사 텍스트만 | speaker, startAt/endAt, 성별, 파일명 등을 예측 특징으로 사용하지 않음 |\n| 9 | 지정 9개 symptom만 처리, 다른 symptom만 제거하고 샘플은 유지 | 9차원 다중 레이블; 빈 정답과 빈 예측도 허용 |\n| 10 | 클래스별 F1을 평균 | 9개 증상의 Macro F1을 선택·보고 지표로 사용 |\n| 11 | 표준 inference.py 명령 한 번으로 CSV 생성 | audio_dir/label_dir/ckpt_path/output 인터페이스; CSV는 label file name, symptom |\n| 12 | 학습 로그 노트북·가중치·모델 코드·requirements 등 제출 | standalone 노트북과 runtime/submission 소스·완료 가중치·기록 보관 |\n\n공개 pretrained 모델의 원래 사전학습 코퍼스를 대회 표본으로 추가하는 것은 아닙니다. 문제 p5의 공개 사전학습 모델 허용 범위에서\n준비한 가중치만 사용하며, 추가적인 원문 코퍼스 수집이나 외부 데이터를 이용한 적응 학습은 수행하지 않습니다.\n모델 준비 단계에는 대회 전사·정답 경로를 전달하지 않고 준비 후 학습·추론은 로컬 파일만 사용합니다.\n\n최종 제출에는 결과 PPT와 **실행 로그가 남은** 노트북도 필요합니다. 소스 생성만으로 실제 GPU 학습·측정·제출이 끝난 것은 아닙니다.\n', 'ver4_runtime/README.md': '# Mission 3 ver4 실행 안내\n\n실행 파일은 [src/mission3_ver4.ipynb](../mission3_ver4.ipynb)입니다. 이 노트북 하나만 Colab에 올리면 필요한 모델·학습·추론 코드를 생성합니다. 이 폴더의 `.py` 파일은 같은 기능을 로컬이나 제출 환경에서 실행하기 위한 코드입니다.\n\n## 기존 결과와 개선 근거\n\n| 실행 버전 | 실제 Validation Macro F1 | 근거 |\n|---|---:|---|\n| `mission3_all_in_one_ver2` → results/mission3/ver1 | 0.600123 | `results/mission3/ver1/mission3_metrics.json` |\n| `mission3_ver3` → results/mission3/ver3 | 0.626015 | `results/mission3/ver3/mission3_ver3_metrics.json` |\n| ver4 | 아직 미측정 | 전체 GPU 학습 후 평가 필요 |\n\nver3의 Training 내부 F1은 3 epoch에서 0.644361로 최고였고, 15 epoch에는 0.601536으로 하락했습니다. 학습 손실은 계속 감소해 과적합 징후가 있습니다. ver4는 별도 stopping 집합의 Macro AP로 최적 epoch를 선택하고 조기 종료합니다. AP는 임계값 없이 예측 순위를 평가하며, 최종 대회 지표는 그대로 **9개 증상의 Macro F1**입니다.\n\nTraining 29,200개를 저장된 ver3 tokenizer로 직접 측정했을 때 384토큰을 넘는 대화는 3,292개(11.27%), 512토큰을 넘는 대화는 788개(2.70%)였습니다. 특수 토큰을 포함한 최대 길이는 873토큰입니다. ver4는 긴 대화의 뒷부분도 처리하도록 설계했습니다.\n\n**현재 작성 환경에는 GPU가 없어 ver4 전체 학습과 Validation 성능은 확인하지 못했습니다.** 작은 CPU 모델을 이용한 실행 검증은 코드 경로 확인용입니다. 기존 버전보다 성능이 높다는 판단은 전체 학습·추론이 끝난 뒤 실제 점수로 해야 합니다.\n\n## 모델과 데이터 분리\n\n- `klue/roberta-large`를 5개 fold에서 각각 학습하고 예측 확률을 평균냅니다.\n- 한 window는 최대 512토큰이며 인접 window는 64토큰이 겹칩니다. 마지막 window까지 유지하고, 증상별 attention으로 대화 전체를 모은 뒤 **통화당 한 번** BCE 손실을 계산합니다.\n- 문자 2–5그램과 단어 1–2그램 TF-IDF + 증상별 Logistic Regression을 함께 학습합니다.\n- 증상마다 신경망/TF-IDF의 혼합 비율과 판정 임계값을 Training 내부에서 결정합니다. 어떤 통화도 최소 1개 증상을 출력하도록 강제하지 않습니다.\n\n서울 Training을 약 90% development와 10% calibration으로 먼저 나눕니다. Development 내부 5-fold는 원래 Training 기준으로 각 fold마다 약 72% fit, 18% stopping입니다. 동일한 정규화 대화는 같은 그룹에 두어 분할을 넘지 않게 합니다.\n\nCalibration 10%는 신경망 학습, TF-IDF 학습, epoch 선택에 전혀 사용하지 않습니다. TF-IDF는 development 90%에만 fit합니다. 완성된 5개 신경망의 평균 확률과 TF-IDF 확률을 calibration에 적용한 뒤, 그 안에서 다시 5-fold 교차 보정하여 혼합 비율·임계값 선택의 일반화 점수를 계산합니다. 배포용 보정값은 calibration 전체로 결정합니다. **보정 후 기본 모델을 다시 학습하지 않고 같은 모델들을 그대로 추론에 사용합니다.** 공식 Validation은 이 과정에 들어가지 않습니다.\n\n주요 기본값은 `pipeline.py`의 `DEFAULT_CONFIG`에 있습니다.\n\n| 설정 | 기본값 |\n|---|---|\n| 모델 / fold | `klue/roberta-large` / 5 |\n| 최대 epoch / 조기 종료 patience | 6 / 2 |\n| 통화 batch / gradient accumulation | 2 / 16 |\n| encoder / head learning rate | `1.5e-5` / `1e-4` |\n| 문자 / 단어 최대 특징 수 | 120,000 / 40,000 |\n\n## Colab에서 실행\n\n1. [mission3_ver4.ipynb](../mission3_ver4.ipynb)를 Colab에 업로드합니다.\n2. GPU 런타임을 선택합니다. 기본 large 모델의 5-fold 학습에는 **A100 또는 L4를 권장**합니다. 사용 가능한 장치에 따라 시간과 메모리 사용량은 달라집니다.\n3. 제공 데이터의 압축을 풀고 `TL_서울_구급`, `VL_서울_구급` 폴더 이름을 유지합니다. Mission 3은 JSON만 읽습니다.\n4. 위에서부터 모든 셀을 실행하고 Google Drive를 연결합니다. 노트북은 `/content`를 먼저 탐색하고 Drive의 알려진 데이터 경로를 확인합니다. 자동 탐색에 실패하면 노트북의 데이터 경로 설정에 실제 폴더를 지정합니다.\n5. 학습·보정이 끝난 뒤 Validation 추론과 평가 셀을 실행합니다. 실행 로그가 남은 노트북도 저장합니다.\n\n기본 결과 폴더는 Colab에서 `/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver4`, 로컬에서 프로젝트의 `results/mission3/ver4`입니다. 신경망 가중치 5개를 저장하므로 Drive 여유 공간도 확인합니다. 패키지와 최초 공개 pretrained 모델 다운로드에는 인터넷 연결이 필요하며, 학습 완료 후에는 저장된 구성으로 오프라인 추론할 수 있습니다.\n\n## 로컬 또는 제출 환경에서 실행\n\n다음 명령은 `DCC_AIdeas` 프로젝트 루트 기준입니다. 전체 학습 명령은 CUDA GPU를 요구합니다.\n\n```bash\npython -m pip install -r src/mission3_ver4/requirements.txt\n\npython src/mission3_ver4/train.py \\\n  --label_dir "DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급" \\\n  --output_dir results/mission3/ver4\n\npython src/mission3_ver4/inference.py \\\n  --audio_dir "DATA/대학부 데이터/Validation/1.원천데이터/VS_서울_구급" \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --ckpt_path results/mission3/ver4/mission3_ver4_model.ckpt \\\n  --output results/mission3/ver4/mission3.csv\n\npython src/mission3_ver4/evaluate.py \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --prediction results/mission3/ver4/mission3.csv \\\n  --output results/mission3/ver4/validation_metrics.json\n```\n\n`--audio_dir`는 주최 측 인터페이스를 맞추기 위해 받으며 음성 파일을 읽지 않습니다. 추론은 `--device cpu`도 지원하지만 large 모델 5개를 처리하므로 GPU보다 느릴 수 있습니다. 학습 설정은 `train.py --config 설정.json`으로 변경할 수 있습니다. 설정을 바꿀 때는 새 `--output_dir`를 사용합니다.\n\n## 결과 읽기와 파일 보관\n\n| 파일 | 의미 |\n|---|---|\n| `mission3_ver4_metrics.json` | Training 내부 보정 점수, 증상별 교차 보정 F1, 최종 보정값 |\n| `mission3_ver4_history.json` | 각 fold의 epoch별 학습·stopping 기록 |\n| `validation_metrics.json` | 위 평가 명령으로 만든 공식 Validation Macro F1와 증상별 점수 |\n| `mission3.csv` | `label file name,symptom` 형식의 예측 결과 |\n| `run.json`, `splits.json` | 실행 설정·데이터/코드 식별값과 분할 파일 목록 |\n| `submission/` | 실행 당시 `.py` 파일, requirements와 실행 안내 사본 |\n\n`calibration_crossfit_macro_f1`는 각 보정 평가 샘플의 정답을 보정값 선택에서 제외하여 계산한 **Training 내부 점수**입니다. `calibration_fit_macro_f1`는 같은 calibration 데이터에서 보정값을 정하고 다시 평가한 점수라 낙관적일 수 있습니다. 둘을 Validation 점수로 보고하면 안 됩니다. 공식 Validation 평가를 모델 선택이나 임계값 재조정에 반복 사용하지 않습니다.\n\n**`mission3_ver4_model.ckpt`는 구성 목록과 보정값을 담은 manifest입니다. 이 파일 하나만 옮기면 추론할 수 없습니다.** 결과 디렉터리 전체를 함께 보관하거나 최소한 다음 구성을 유지합니다.\n\n```text\nver4/\n├─ mission3_ver4_model.ckpt\n├─ fold_1.ckpt ... fold_5.ckpt\n├─ tokenizer/\n├─ sparse.joblib\n└─ submission/\n```\n\nCSV의 symptom 값은 기존 결과와 같은 문자열 목록 표현입니다. 예를 들어 `[]`, `[\'구토\', \'복통\']`처럼 저장하며 CSV 인용 처리는 코드가 담당합니다. 주최 측 sample submission에서 더 구체적인 형식을 제공하면 해당 규격도 확인합니다.\n\n## 중단 후 다시 실행\n\n같은 데이터·설정·코드와 같은 결과 폴더로 다시 실행하면 `fold_N.ckpt`와 `fold_N_done.json`이 모두 존재하는 **완료된 fold만 재사용**합니다. 실행 도중 끊긴 fold는 처음부터 다시 학습합니다. 마지막 미니배치나 optimizer 상태에서 이어가는 기능은 아닙니다.\n\n데이터·설정·코드의 식별값이 기존 실행과 다르면 혼합된 결과를 만들지 않도록 중단합니다. 이 경우 새 결과 폴더를 지정합니다. 메모리가 부족하면 통화 batch와 window batch를 줄이는 설정을 검토하고 새 폴더에서 시작합니다.\n\n## 대회 조건과 참고 자료\n\n판단 기준은 [docs의 출제 PDF](../../docs/0820_DCC%20대학부_예선_출제문제.pdf) 5쪽, 9–12쪽입니다. 서울 Training만 학습하고 양측의 전사 `text`만 입력으로 사용합니다. `speaker`, 시간, 주소, 성별, 정답 라벨은 입력 특징에서 제외합니다. 대상은 `고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란`이며, 나머지 라벨만 제거하고 통화는 유지합니다. 외부 추가 데이터와 상용 AI API를 사용하지 않습니다.\n\n최종 제출에는 추론 코드, 학습 로그 포함 노트북, 학습 완료 가중치, 모델 코드, requirements 및 결과 정리 PPT가 필요합니다. 이 구현의 학습을 완료하고 로그·가중치를 보관해야 하며 PPT는 별도로 준비합니다.\n\n공식 구현 참고: [KLUE RoBERTa large](https://huggingface.co/klue/roberta-large), [Transformers tokenizer](https://huggingface.co/docs/transformers/en/main_classes/tokenizer), [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html). 모델 로딩·토큰 처리·혼합 정밀도 API 참고 자료이며 Mission 3의 성능 향상을 보장하는 근거는 아닙니다.\n', 'ver4_runtime/__init__.py': '"""Unmodified ver4 runtime retained for compatible inference and run verification."""\n', 'ver4_runtime/calibration.py': '"""Training-only grouped calibration for nine independent symptom labels."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nif __package__:\n    from .data import TARGET_SYMPTOMS\nelse:\n    from data import TARGET_SYMPTOMS\n\nWEIGHT_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)\nALL_NEGATIVE_THRESHOLD = float(np.nextafter(1.0, np.inf))\n\n\ndef _probabilities(values, name):\n    values = np.asarray(values, dtype=np.float64)\n    if values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS):\n        raise ValueError(f"{name} must have shape (samples, {len(TARGET_SYMPTOMS)})")\n    if not len(values) or not np.isfinite(values).all():\n        raise ValueError(f"{name} must be nonempty and finite")\n    if np.any((values < 0) | (values > 1)):\n        raise ValueError(f"{name} must contain probabilities in [0, 1]")\n    return values\n\n\ndef _inputs(y_true, neural_probs, sparse_probs):\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    truth = np.asarray(y_true)\n    if truth.shape != neural.shape or sparse.shape != neural.shape:\n        raise ValueError("truth, neural_probs and sparse_probs must have identical shapes")\n    if not np.isin(truth, (0, 1)).all():\n        raise ValueError("y_true must contain only binary labels")\n    return truth.astype(np.int8), neural, sparse\n\n\ndef _best_threshold(truth, probabilities):\n    """Exact F1 maximum for >= decisions; tied probabilities stay together."""\n    positives = int(truth.sum())\n    if positives == 0:\n        return ALL_NEGATIVE_THRESHOLD, 0.0\n    if positives == len(truth):\n        return 0.0, 1.0\n    order = np.argsort(-probabilities, kind="stable")\n    sorted_probs = probabilities[order]\n    # A threshold can include a complete tied block, never part of one.\n    boundaries = np.r_[np.flatnonzero(sorted_probs[:-1] != sorted_probs[1:]), len(truth) - 1]\n    true_positives = np.cumsum(truth[order])[boundaries]\n    scores = 2.0 * true_positives / (positives + boundaries + 1)\n    thresholds = np.zeros(len(boundaries), dtype=np.float64)\n    has_next = boundaries < len(truth) - 1\n    upper = sorted_probs[boundaries[has_next]]\n    lower = sorted_probs[boundaries[has_next] + 1]\n    midpoint = lower + (upper - lower) / 2\n    # Adjacent floating-point values may have no representable midpoint.\n    thresholds[has_next] = np.where(midpoint > lower, midpoint, upper)\n    best = np.flatnonzero(scores == scores.max())\n    # An explicit deterministic tie break preserves the exact maximum.\n    winner = min(best, key=lambda i: (abs(thresholds[i] - 0.5), -thresholds[i]))\n    return float(thresholds[winner]), float(scores[winner])\n\n\ndef fit_calibrator(y_true, neural_probs, sparse_probs):\n    """Select each label\'s blend and threshold using supplied training labels."""\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    weights, thresholds = [], []\n    for column in range(len(TARGET_SYMPTOMS)):\n        candidates = []\n        for weight in WEIGHT_GRID:\n            blended = weight * neural[:, column] + (1.0 - weight) * sparse[:, column]\n            threshold, score = _best_threshold(truth[:, column], blended)\n            candidates.append((score, -abs(weight - 0.5), weight, threshold))\n        _, _, weight, threshold = max(candidates)\n        weights.append(float(weight))\n        thresholds.append(float(threshold))\n    return {"neural_weights": weights, "thresholds": thresholds}\n\n\ndef apply_calibrator(neural_probs, sparse_probs, calibrator):\n    """Return blended probabilities; use >= stored thresholds for labels."""\n    neural = _probabilities(neural_probs, "neural_probs")\n    sparse = _probabilities(sparse_probs, "sparse_probs")\n    if neural.shape != sparse.shape:\n        raise ValueError("neural_probs and sparse_probs must have identical shapes")\n    weights = np.asarray(calibrator["neural_weights"], dtype=np.float64)\n    thresholds = np.asarray(calibrator["thresholds"], dtype=np.float64)\n    expected = (len(TARGET_SYMPTOMS),)\n    if weights.shape != expected or thresholds.shape != expected:\n        raise ValueError("calibrator must contain one weight and threshold per symptom")\n    if not np.isfinite(weights).all() or np.any((weights < 0) | (weights > 1)):\n        raise ValueError("calibrator weights must be finite and in [0, 1]")\n    if not np.isfinite(thresholds).all() or np.any((thresholds < 0) | (thresholds > ALL_NEGATIVE_THRESHOLD)):\n        raise ValueError("calibrator thresholds must be finite and in [0, nextafter(1, inf)]")\n    return weights * neural + (1.0 - weights) * sparse\n\n\ndef _grouped_splits(truth, groups, seed, n_splits):\n    groups = np.asarray(groups)\n    if groups.ndim != 1 or len(groups) != len(truth):\n        raise ValueError("groups must contain one group ID per sample")\n    if not isinstance(n_splits, (int, np.integer)) or n_splits < 2:\n        raise ValueError("n_splits must be an integer >= 2")\n    unique_groups, inverse = np.unique(groups, return_inverse=True)\n    if len(unique_groups) < 2:\n        raise ValueError("cross-fit calibration requires at least two distinct groups")\n    n_splits = min(n_splits, len(unique_groups))\n    group_targets = np.zeros((len(unique_groups), truth.shape[1]), dtype=np.int8)\n    np.maximum.at(group_targets, inverse, truth)\n    splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n    for train_groups, test_groups in splitter.split(np.zeros(len(unique_groups)), group_targets):\n        train = np.flatnonzero(np.isin(inverse, train_groups))\n        test = np.flatnonzero(np.isin(inverse, test_groups))\n        if not len(train) or not len(test):\n            raise ValueError("grouped calibration produced an empty fold")\n        yield train, test\n\n\ndef calibrate_crossfit(y_true, neural_probs, sparse_probs, groups, seed=42, n_splits=5):\n    """Cross-fit within Training calibration data, then fit deployment settings.\n\n    Base-model predictions must themselves come from samples excluded from\n    base-model training and model selection. Never pass official Validation.\n    """\n    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)\n    crossfit_predictions = np.zeros_like(truth)\n    seen = np.zeros(len(truth), dtype=np.int8)\n    fold_counts = []\n    for train, test in _grouped_splits(truth, groups, seed, n_splits):\n        calibrator = fit_calibrator(truth[train], neural[train], sparse[train])\n        probabilities = apply_calibrator(neural[test], sparse[test], calibrator)\n        crossfit_predictions[test] = probabilities >= np.asarray(calibrator["thresholds"])\n        seen[test] += 1\n        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})\n    if not np.all(seen == 1):\n        raise RuntimeError("each calibration sample must be scored exactly once")\n    final_calibrator = fit_calibrator(truth, neural, sparse)\n    fitted = apply_calibrator(neural, sparse, final_calibrator) >= np.asarray(final_calibrator["thresholds"])\n    precision, recall, f1, support = precision_recall_fscore_support(\n        truth, crossfit_predictions, average=None, zero_division=0\n    )\n    report = {\n        "score_scope": "Training-only calibration; cross-fit estimates calibration generalization, not official Validation.",\n        "calibration_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),\n        "calibration_crossfit_macro_f1": float(f1.mean()),\n        "per_symptom_crossfit": {\n            symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}\n            for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)\n        },\n        "calibration_samples": int(len(truth)),\n        "calibration_groups": int(len(np.unique(groups))),\n        "crossfit_splits": len(fold_counts),\n        "crossfit_fold_counts": fold_counts,\n    }\n    return final_calibrator, report\n', 'ver4_runtime/data.py': '"""Mission 3: 모델 입력은 양측 발화의 text만, symptom은 학습/평가 정답만."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport unicodedata\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """입력/정답과 무관하게 동일하게 전처리하며 발화 순서를 유지한다."""\n    utterances = annotation.get("utterances")\n    if not isinstance(utterances, list):\n        utterances = annotation.get("utterences", [])\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if isinstance(utterance, dict) and isinstance(utterance.get("text"), str):\n            text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n            if text:\n                texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(x, str) for x in raw):\n        raise ValueError("symptom must be a string or a list of strings")\n    found = {x.strip() for x in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    # 파일명, 날짜, speaker 등은 모델 특징에도 중복 판단에도 사용하지 않는다.\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir: Path, include_targets: bool = False, workers: int = 8):\n    paths = sorted(Path(label_dir).rglob("*.json"))\n    if not paths:\n        raise ValueError(f"JSON 파일이 없습니다: {label_dir}")\n    if len({p.name for p in paths}) != len(paths):\n        raise ValueError("동일한 JSON 파일명이 중복됩니다. 제출 CSV의 식별자가 모호합니다.")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"JSON object가 아닙니다: {path}")\n        # 추론 경로에서는 target_vector를 호출하거나 symptom에 접근하지 않는다.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef grouped_targets(texts, targets):\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse = np.unique(groups, return_inverse=True)\n    aggregate = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    np.maximum.at(aggregate, inverse, targets)\n    return groups, inverse, aggregate\n\n\ndef make_splits(texts, targets, calibration_size: float, n_folds: int, seed: int):\n    """전체 Training의 보정 집합은 모든 모델 fit/epoch 선택에서 분리한다."""\n    if not 0 < calibration_size < 0.5 or n_folds < 2:\n        raise ValueError("calibration_size must be in (0, .5); n_folds >= 2")\n    groups, inverse, aggregate = grouped_targets(texts, targets)\n    group_ids = np.arange(len(aggregate))\n    outer = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=calibration_size, random_state=seed)\n    dev_groups, cal_groups = next(outer.split(group_ids, aggregate))\n    development = np.flatnonzero(np.isin(inverse, dev_groups))\n    calibration = np.flatnonzero(np.isin(inverse, cal_groups))\n    splitter = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed + 1)\n    folds = []\n    for train_g, stop_g in splitter.split(dev_groups, aggregate[dev_groups]):\n        train_indices = np.flatnonzero(np.isin(inverse, dev_groups[train_g]))\n        stop_indices = np.flatnonzero(np.isin(inverse, dev_groups[stop_g]))\n        folds.append((train_indices, stop_indices))\n    for name, indices in [("calibration", calibration), ("development", development),\n                          *[(f"fold{i}/{part}", ids) for i, pair in enumerate(folds)\n                            for part, ids in zip(("fit", "stop"), pair)]]:\n        counts = targets[indices].sum(axis=0)\n        if len(indices) == 0 or np.any(counts == 0) or np.any(counts == len(indices)):\n            raise ValueError(f"{name}: 각 증상의 양성/음성이 필요합니다. 데이터/분할 수를 확인하세요.")\n    return development, calibration, folds, groups\n\n\ndef write_json(path: Path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef write_predictions(paths, probabilities, thresholds, output: Path):\n    probabilities = np.asarray(probabilities)\n    if probabilities.shape != (len(paths), len(TARGET_SYMPTOMS)) or not np.isfinite(probabilities).all():\n        raise ValueError("Invalid prediction shape or non-finite probabilities")\n    predicted = probabilities >= np.asarray(thresholds)\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predicted):\n            # 기존 제출 CSV와 같은 Python string-list 표현. 빈 list도 허용한다.\n            writer.writerow([Path(path).name, str([s for s, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    return predicted.astype(np.int8)\n\n\ndef score_predictions(targets, predictions):\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i])}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n', 'ver4_runtime/evaluate.py': '"""Score frozen CSV predictions after inference, without changing the model."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .data import write_json\n    from .pipeline import evaluate_csv\nelse:\n    from data import write_json\n    from pipeline import evaluate_csv\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--prediction", required=True, type=Path)\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n    report = evaluate_csv(args.label_dir, args.prediction)\n    if args.output:\n        write_json(args.output, report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'ver4_runtime/inference.py': '"""Official one-command inference. Mission 3 accepts but does not read audio_dir."""\nimport argparse\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import predict_pipeline\nelse:\n    from pipeline import predict_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--audio_dir", required=True, help="Organizer interface; unused text-only task")\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--ckpt_path", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--device", choices=["cpu", "cuda"])\n    parser.add_argument("--batch_size", type=int)\n    args = parser.parse_args()\n    predict_pipeline(args.ckpt_path, args.label_dir, args.output, args.device, args.batch_size)\n\n\nif __name__ == "__main__":\n    main()\n', 'ver4_runtime/neural.py': '"""Document-level multi-label transformer; every dialogue window is retained."""\n\nfrom __future__ import annotations\n\nimport math\nimport os\nimport random\nimport re\nimport tempfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import average_precision_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup\n\n\nNUM_LABELS = 9\n\n\ndef tokenize_dialogues(texts, tokenizer, max_length=512, stride=64):\n    """Return documents of token windows, including the last window of long calls.\n\n    Only ordinary tokenizer special tokens are used. The stride is an overlap in\n    tokens, not an upper bound on the number of windows or dialogue length.\n    """\n    usable_length = max_length - tokenizer.num_special_tokens_to_add(pair=False)\n    if usable_length < 1 or not 0 <= stride < usable_length:\n        raise ValueError("max_length must fit special tokens; 0 <= stride < usable length")\n    texts = list(texts)\n    documents = [[] for _ in texts]\n    for start in range(0, len(texts), 128):\n        encoded = tokenizer(\n            texts[start:start + 128], truncation=True, max_length=max_length,\n            stride=stride, return_overflowing_tokens=True, padding=False,\n            return_attention_mask=True,\n        )\n        if "overflow_to_sample_mapping" not in encoded:\n            raise ValueError("Use a fast tokenizer (AutoTokenizer(..., use_fast=True))")\n        for window, sample in enumerate(encoded["overflow_to_sample_mapping"]):\n            documents[start + int(sample)].append({\n                key: encoded[key][window]\n                for key in tokenizer.model_input_names if key in encoded\n            })\n    if any(not windows for windows in documents):\n        raise ValueError("Tokenizer returned a document without any windows")\n    return documents\n\n\nclass DialogueDataset(Dataset):\n    def __init__(self, documents, targets=None):\n        self.documents, self.targets = documents, targets\n\n    def __len__(self):\n        return len(self.documents)\n\n    def __getitem__(self, index):\n        item = {"windows": self.documents[index]}\n        if self.targets is not None:\n            item["labels"] = self.targets[index]\n        return item\n\n\nclass DialogueCollator:\n    """Flatten windows for the encoder while keeping exactly one target per call."""\n\n    def __init__(self, tokenizer):\n        self.tokenizer = tokenizer\n\n    def __call__(self, examples):\n        windows, doc_indices = [], []\n        for index, example in enumerate(examples):\n            if not example["windows"]:\n                raise ValueError("A document must contain at least one token window")\n            windows.extend(example["windows"])\n            doc_indices.extend([index] * len(example["windows"]))\n        batch = dict(self.tokenizer.pad(windows, padding=True, return_tensors="pt"))\n        batch["doc_indices"] = torch.tensor(doc_indices, dtype=torch.long)\n        if "labels" in examples[0]:\n            batch["labels"] = torch.from_numpy(np.asarray(\n                [example["labels"] for example in examples], dtype=np.float32,\n            ))\n        return batch\n\n\nclass DialogueClassifier(nn.Module):\n    """Label-specific attention pools complete dialogues before computing BCE."""\n\n    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.2,\n                 window_batch_size=8):\n        super().__init__()\n        self.encoder = encoder\n        self.window_batch_size = int(window_batch_size)\n        if self.window_batch_size < 1:\n            raise ValueError("window_batch_size must be positive")\n        hidden = encoder.config.hidden_size\n        self.projection = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.Tanh())\n        self.dropout = nn.Dropout(dropout)\n        self.attention_queries = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_weights = nn.Parameter(torch.empty(num_labels, hidden))\n        self.label_bias = nn.Parameter(torch.zeros(num_labels))\n        nn.init.normal_(self.attention_queries, std=0.02)\n        nn.init.normal_(self.label_weights, std=0.02)\n\n    def forward(self, doc_indices, **inputs):\n        representations = []\n        for start in range(0, len(doc_indices), self.window_batch_size):\n            window_inputs = {key: value[start:start + self.window_batch_size]\n                             for key, value in inputs.items()}\n            hidden = self.encoder(**window_inputs).last_hidden_state\n            mask = window_inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n            mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)\n            representations.append(self.projection(torch.cat([hidden[:, 0], mean], -1)))\n        windows = self.dropout(torch.cat(representations))\n        logits = []\n        # Each label learns which part of a potentially long call provides evidence.\n        for document_index in range(int(doc_indices.max().item()) + 1):\n            selected = windows[doc_indices == document_index]\n            scores = selected @ self.attention_queries.T / math.sqrt(selected.shape[-1])\n            attention = torch.softmax(scores.float(), dim=0).to(selected.dtype)\n            pooled = attention.T @ selected\n            logits.append((self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias)\n        return torch.stack(logits)\n\n\ndef _device(config):\n    device = torch.device(config.get("device", "cuda" if torch.cuda.is_available() else "cpu"))\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA was requested but is unavailable. Select a GPU runtime.")\n    return device\n\n\ndef _amp(device):\n    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16\n    return torch.amp.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")\n\n\ndef _loader(documents, targets, tokenizer, batch_size, shuffle=False, seed=0):\n    if int(batch_size) < 1:\n        raise ValueError("batch_size must be positive")\n    generator = torch.Generator().manual_seed(seed)\n    return DataLoader(\n        DialogueDataset(documents, targets), batch_size=int(batch_size),\n        shuffle=shuffle, collate_fn=DialogueCollator(tokenizer),\n        num_workers=0, generator=generator,\n    )\n\n\n@torch.inference_mode()\ndef _predict(model, loader, device):\n    model.eval()\n    probabilities = []\n    for batch in loader:\n        batch.pop("labels", None)\n        batch = {key: value.to(device) for key, value in batch.items()}\n        with _amp(device):\n            logits = model(**batch)\n        probabilities.append(logits.float().sigmoid().cpu().numpy())\n    return np.concatenate(probabilities) if probabilities else np.empty((0, NUM_LABELS), dtype=np.float32)\n\n\ndef _optimizer(model, config):\n    depth = int(getattr(model.encoder.config, "num_hidden_layers", 0))\n    decay = float(config.get("layerwise_lr_decay", 0.9))\n    if not 0 < decay <= 1:\n        raise ValueError("layerwise_lr_decay must be in (0, 1]")\n    groups = {}\n    for name, parameter in model.named_parameters():\n        if not parameter.requires_grad:\n            continue\n        if name.startswith("encoder."):\n            match = re.search(r"\\.layer\\.(\\d+)\\.", name)\n            level = int(match.group(1)) + 1 if match else (0 if ".embeddings." in name else depth + 1)\n            lr = float(config.get("learning_rate", 1.5e-5)) * decay ** (depth + 1 - level)\n        else:\n            lr = float(config.get("head_learning_rate", 1e-4))\n        wd = 0.0 if parameter.ndim < 2 or name.endswith("bias") else float(config.get("weight_decay", 0.01))\n        groups.setdefault((lr, wd), []).append(parameter)\n    return torch.optim.AdamW([{"params": params, "lr": lr, "weight_decay": wd}\n                             for (lr, wd), params in groups.items()])\n\n\ndef _atomic_save(checkpoint, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    os.close(descriptor)\n    try:\n        torch.save(checkpoint, temporary)\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef _targets(targets, documents, name):\n    targets = np.asarray(targets, dtype=np.float32)\n    if targets.shape != (len(documents), NUM_LABELS) or not len(documents):\n        raise ValueError(f"{name} requires nonempty documents and targets of shape (N, {NUM_LABELS})")\n    if not np.isin(targets, [0, 1]).all():\n        raise ValueError(f"{name} targets must contain only 0 and 1")\n    return targets\n\n\ndef train_model(train_documents, train_targets, stop_documents, stop_targets,\n                tokenizer, config: dict, checkpoint_path: Path, seed: int):\n    """Train on documents; select best epoch only by stopping-set macro AP.\n\n    This function does not tune thresholds and never uses calibration targets.\n    GPU enforcement for production belongs to the orchestration entry point.\n    """\n    train_targets = _targets(train_targets, train_documents, "train")\n    stop_targets = _targets(stop_targets, stop_documents, "stop")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    device = _device(config)\n    max_epochs, patience = int(config.get("max_epochs", 8)), int(config.get("patience", 2))\n    accumulation = int(config.get("gradient_accumulation_steps", 4))\n    if min(max_epochs, patience, accumulation) < 1:\n        raise ValueError("max_epochs, patience and gradient_accumulation_steps must be positive")\n    model_name = config.get("model_name", "klue/roberta-large")\n    encoder = AutoModel.from_pretrained(model_name)\n    if config.get("gradient_checkpointing", True):\n        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    if hasattr(encoder.config, "use_cache"):\n        encoder.config.use_cache = False\n    architecture = {"num_labels": NUM_LABELS, "dropout": float(config.get("dropout", 0.2)),\n                    "window_batch_size": int(config.get("window_batch_size", 8))}\n    model = DialogueClassifier(encoder, **architecture).to(device)\n    train_loader = _loader(train_documents, train_targets, tokenizer,\n                           config.get("batch_size", 2), shuffle=True, seed=seed)\n    stop_loader = _loader(stop_documents, None, tokenizer, config.get("inference_batch_size", 4))\n    positives = train_targets.sum(axis=0)\n    if (positives == 0).any():\n        raise ValueError(f"Training split has no positive examples for labels {np.flatnonzero(positives == 0).tolist()}")\n    pos_weight = np.clip(((len(train_targets) - positives) / positives).clip(min=1)\n                         ** float(config.get("pos_weight_power", 0.5)), 1, 5)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.as_tensor(pos_weight, device=device))\n    optimizer = _optimizer(model, config)\n    total_steps = math.ceil(len(train_loader) / accumulation) * max_epochs\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer, num_warmup_steps=int(total_steps * float(config.get("warmup_ratio", 0.1))),\n        num_training_steps=total_steps,\n    )\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())\n    valid_labels = stop_targets.sum(axis=0) > 0\n    if not valid_labels.any():\n        raise ValueError("Stopping split has no positive labels; macro AP cannot select an epoch")\n    if not valid_labels.all():\n        print(f"Warning: stop AP excludes labels without positives: {np.flatnonzero(~valid_labels).tolist()}", flush=True)\n    best_ap, best_epoch, stale, history = -1.0, 0, 0, []\n    for epoch in range(1, max_epochs + 1):\n        epoch_started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss_sum, document_count = 0.0, 0\n        for step, batch in enumerate(train_loader):\n            labels = batch.pop("labels").to(device)\n            batch = {key: value.to(device) for key, value in batch.items()}\n            # Match a single full document batch, including a shorter last batch.\n            group_start = (step // accumulation) * accumulation\n            group_documents = min(accumulation * train_loader.batch_size,\n                                  len(train_documents) - group_start * train_loader.batch_size)\n            with _amp(device):\n                loss = criterion(model(**batch).float(), labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Non-finite training loss at epoch={epoch}, batch={step}")\n            scaler.scale(loss * len(labels) / group_documents).backward()\n            loss_sum += loss.detach().item() * len(labels)\n            document_count += len(labels)\n            if (step + 1) % accumulation == 0 or step + 1 == len(train_loader):\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            log_every = int(config.get("log_every", 100))\n            if log_every > 0 and (step + 1) % log_every == 0:\n                elapsed = (time.monotonic() - epoch_started) / 60\n                print(f"seed={seed} epoch={epoch} batch={step + 1}/{len(train_loader)} "\n                      f"loss={loss_sum / document_count:.5f} minutes={elapsed:.1f}", flush=True)\n        probabilities = _predict(model, stop_loader, device)\n        if not np.isfinite(probabilities).all():\n            raise FloatingPointError("Stopping predictions contain non-finite values")\n        score = float(average_precision_score(stop_targets[:, valid_labels],\n                                              probabilities[:, valid_labels], average="macro"))\n        row = {"epoch": epoch, "train_loss": loss_sum / document_count,\n               "stop_macro_ap": score, "stop_labels_with_positives": int(valid_labels.sum()),\n               "minutes": (time.monotonic() - epoch_started) / 60}\n        history.append(row)\n        print(f"seed={seed} epoch={epoch}/{max_epochs} loss={row[\'train_loss\']:.5f} "\n              f"stop_AP={score:.5f} minutes={row[\'minutes\']:.1f}", flush=True)\n        if score > best_ap + 1e-6:\n            best_ap, best_epoch, stale = score, epoch, 0\n            _atomic_save({\n                "format_version": 1, "encoder_config": encoder.config.to_dict(),\n                "architecture": architecture, "model_name": str(model_name),\n                "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},\n                "epoch": epoch, "stop_ap": score, "seed": int(seed),\n            }, checkpoint_path)\n        else:\n            stale += 1\n            if stale >= patience:\n                break\n    return {"best_epoch": best_epoch, "best_stop_ap": best_ap, "history": history}\n\n\ndef predict_checkpoint(checkpoint_path, documents, tokenizer, config):\n    """Rebuild from the saved config and weights; no pretrained network fetch."""\n    device = _device(config)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if checkpoint.get("format_version") != 1:\n        raise ValueError(f"Unsupported neural checkpoint format: {checkpoint_path}")\n    encoder_data = dict(checkpoint["encoder_config"])\n    model_type = encoder_data.pop("model_type")\n    encoder_config = AutoConfig.for_model(model_type, **encoder_data)\n    architecture = dict(checkpoint["architecture"])\n    architecture["window_batch_size"] = int(config.get("window_batch_size", architecture["window_batch_size"]))\n    model = DialogueClassifier(AutoModel.from_config(encoder_config), **architecture)\n    model.load_state_dict(checkpoint["model_state"], strict=True)\n    del checkpoint\n    model.to(device)\n    loader = _loader(documents, None, tokenizer, config.get("inference_batch_size", 4))\n    predictions = _predict(model, loader, device)\n    if not np.isfinite(predictions).all():\n        raise FloatingPointError("Inference predictions contain non-finite values")\n    return predictions\n', 'ver4_runtime/pipeline.py': '"""Training-only model selection, ensemble calibration, and portable inference."""\nfrom __future__ import annotations\n\nimport ast\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import FeatureUnion\nfrom transformers import AutoTokenizer\n\nif __package__:\n    from .calibration import apply_calibrator, calibrate_crossfit\n    from .data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from .neural import predict_checkpoint, tokenize_dialogues, train_model\nelse:\n    from calibration import apply_calibrator, calibrate_crossfit\n    from data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions\n    from neural import predict_checkpoint, tokenize_dialogues, train_model\n\n\nDEFAULT_CONFIG = {\n    "model_name": "klue/roberta-large",\n    "seed": 42,\n    "n_folds": 5,\n    "calibration_size": 0.10,\n    "calibration_folds": 5,\n    "max_length": 512,\n    "stride": 64,\n    "max_epochs": 6,\n    "patience": 2,\n    "batch_size": 2,\n    "inference_batch_size": 8,\n    "window_batch_size": 2,\n    "gradient_accumulation_steps": 16,\n    "learning_rate": 1.5e-5,\n    "head_learning_rate": 1e-4,\n    "layerwise_lr_decay": 0.95,\n    "weight_decay": 0.01,\n    "warmup_ratio": 0.10,\n    "dropout": 0.20,\n    "pos_weight_power": 0.5,\n    "gradient_checkpointing": True,\n    "device": "cuda",\n    "require_cuda": True,\n    "char_features": 120_000,\n    "word_features": 40_000,\n    "min_df": 2,\n    "logistic_c": 4.0,\n}\nRUNTIME_FILES = ("data.py", "calibration.py", "neural.py", "pipeline.py", "train.py", "inference.py", "evaluate.py")\n\n\ndef fit_sparse(texts, targets, config):\n    """문자 표현과 단어 순서를 보완하는, Training에만 fit한 TF-IDF 모델."""\n    features = FeatureUnion([\n        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5),\n                                max_features=config["char_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), token_pattern=r"(?u)\\b\\w+\\b",\n                                max_features=config["word_features"], min_df=config["min_df"],\n                                sublinear_tf=True, dtype=np.float32)),\n    ])\n    matrix = features.fit_transform(texts)\n    classifiers = []\n    for column, name in enumerate(TARGET_SYMPTOMS):\n        truth = targets[:, column]\n        if len(np.unique(truth)) == 1:\n            classifier = float(truth[0])\n        else:\n            classifier = LogisticRegression(C=config["logistic_c"], class_weight="balanced",\n                                            solver="liblinear", max_iter=1500, random_state=config["seed"])\n            classifier.fit(matrix, truth)\n        classifiers.append(classifier)\n        print(f"TF-IDF fit: {name}", flush=True)\n    return {"features": features, "classifiers": classifiers}\n\n\ndef predict_sparse(model, texts):\n    if not texts:\n        return np.empty((0, len(TARGET_SYMPTOMS)), dtype=np.float32)\n    # 큰 Test 폴더도 전체 sparse matrix를 RAM에 동시에 올리지 않는다.\n    batches = []\n    for start in range(0, len(texts), 1024):\n        matrix = model["features"].transform(texts[start:start + 1024])\n        columns = [np.full(matrix.shape[0], clf) if isinstance(clf, float)\n                   else clf.predict_proba(matrix)[:, 1] for clf in model["classifiers"]]\n        batches.append(np.column_stack(columns))\n    return np.concatenate(batches).astype(np.float32)\n\n\ndef _release_memory():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _torch_save(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef _fingerprint(paths, texts, targets, config):\n    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())\n    for path, text, target in zip(paths, texts, targets):\n        digest.update(json.dumps([path.name, text, target.tolist()], ensure_ascii=False).encode("utf-8"))\n    # 변경된 코드로 기존 fold가 조용히 재사용되는 것을 방지한다.\n    for name in RUNTIME_FILES:\n        path = Path(__file__).parent / name\n        digest.update(path.name.encode())\n        digest.update(path.read_bytes())\n    return digest.hexdigest()\n\n\ndef train_pipeline(train_label_dir, output_dir, config=None):\n    """Validation 인자를 받지 않는다. 보정 후에도 모델을 재학습하지 않는다."""\n    supplied = config or {}\n    unknown = set(supplied) - set(DEFAULT_CONFIG)\n    if unknown:\n        raise ValueError(f"Unknown config keys: {sorted(unknown)}")\n    config = {**DEFAULT_CONFIG, **supplied}\n    train_label_dir, output_dir = Path(train_label_dir).resolve(), Path(output_dir).resolve()\n    if train_label_dir.name != "TL_서울_구급" or "Validation" in train_label_dir.parts:\n        raise ValueError("학습 입력은 서울 Training의 TL_서울_구급 폴더여야 합니다.")\n    if config["require_cuda"] and (config["device"] != "cuda" or not torch.cuda.is_available()):\n        raise RuntimeError("전체 학습에는 GPU가 필요합니다. Colab GPU 런타임에서 실행하세요.")\n    if config["device"] == "cuda":\n        torch.set_float32_matmul_precision("high")\n        print("GPU:", torch.cuda.get_device_name(0), flush=True)\n    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)\n    dev, calibration, folds, groups = make_splits(\n        texts, targets, config["calibration_size"], config["n_folds"], config["seed"],\n    )\n    output_dir.mkdir(parents=True, exist_ok=True)\n    signature = _fingerprint(paths, texts, targets, config)\n    run_path = output_dir / "run.json"\n    if run_path.exists() and json.loads(run_path.read_text(encoding="utf-8"))["signature"] != signature:\n        raise ValueError("출력 폴더에 다른 데이터/설정/코드의 실험이 있습니다. 새 output_dir를 지정하세요.")\n    write_json(run_path, {"signature": signature, "config": config, "training_samples": len(paths)})\n    write_json(output_dir / "splits.json", {\n        "calibration": [paths[i].name for i in calibration],\n        "folds": [{"fit": [paths[i].name for i in fit], "stop": [paths[i].name for i in stop]}\n                  for fit, stop in folds],\n        "seed": config["seed"], "grouping": "SHA256 of whitespace-normalized transcript only",\n    })\n    print(f"Training={len(paths):,}, development={len(dev):,}, calibration={len(calibration):,}", flush=True)\n    tokenizer_dir = output_dir / "tokenizer"\n    if (tokenizer_dir / "tokenizer_config.json").exists():\n        tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True, local_files_only=True)\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast=True)\n        tokenizer.save_pretrained(tokenizer_dir)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    calibration_documents = [documents[i] for i in calibration]\n    counts = np.asarray([len(document) for document in documents])\n    print(f"대화 window 수: 평균={counts.mean():.2f}, 최대={counts.max()}, 긴 대화={int((counts > 1).sum()):,}", flush=True)\n    neural_probs = np.zeros((len(calibration), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    fold_names, histories = [], []\n    for number, (fit, stop) in enumerate(folds, 1):\n        name = f"fold_{number}.ckpt"\n        checkpoint_path, done_path = output_dir / name, output_dir / f"fold_{number}_done.json"\n        print(f"\\nFold {number}/{len(folds)}: fit={len(fit):,}, stop={len(stop):,}", flush=True)\n        if done_path.exists() and checkpoint_path.exists():\n            result = json.loads(done_path.read_text(encoding="utf-8"))\n            print("완료된 fold 재사용 (동일 run signature 확인됨)", flush=True)\n        else:\n            result = train_model([documents[i] for i in fit], targets[fit],\n                                 [documents[i] for i in stop], targets[stop], tokenizer,\n                                 config, checkpoint_path, seed=config["seed"] + number * 1009)\n            write_json(done_path, result)\n        _release_memory()\n        neural_probs += predict_checkpoint(checkpoint_path, calibration_documents, tokenizer, config)\n        _release_memory()\n        fold_names.append(name)\n        histories.append(result)\n    neural_probs /= len(folds)\n    sparse_path = output_dir / "sparse.joblib"\n    if sparse_path.exists():\n        sparse = joblib.load(sparse_path)\n    else:\n        sparse = fit_sparse([texts[i] for i in dev], targets[dev], config)\n        temporary = sparse_path.with_name("sparse.joblib.tmp")\n        joblib.dump(sparse, temporary, compress=3)\n        temporary.replace(sparse_path)\n    sparse_probs = predict_sparse(sparse, [texts[i] for i in calibration])\n    del sparse\n    calibrator, report = calibrate_crossfit(targets[calibration], neural_probs, sparse_probs,\n                                          groups[calibration], seed=config["seed"] + 2,\n                                          n_splits=config["calibration_folds"])\n    # OOF 확률을 final refit에 전용하지 않고, 실제 배포 ensemble의 확률을 보정한다.\n    np.savez_compressed(output_dir / "calibration_predictions.npz", targets=targets[calibration],\n                        neural=neural_probs, sparse=sparse_probs,\n                        filenames=np.asarray([paths[i].name for i in calibration]))\n    versions = {name: importlib.metadata.version(name) for name in\n                ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}\n    manifest = {"format": "mission3_ver4", "version": 1, "target_symptoms": list(TARGET_SYMPTOMS),\n                "config": config, "folds": fold_names, "tokenizer": "tokenizer", "sparse": "sparse.joblib",\n                "calibrator": calibrator, "signature": signature, "versions": versions}\n    checkpoint_path = output_dir / "mission3_ver4_model.ckpt"\n    _torch_save(checkpoint_path, manifest)\n    report.update({"training_samples": len(paths), "development_samples": len(dev),\n                   "model": f"{config[\'model_name\']} {len(folds)}-fold ensemble + char/word TF-IDF LR",\n                   "calibrator": calibrator, "fold_best_epochs": [x["best_epoch"] for x in histories],\n                   "versions": versions, "validation_used_for_training_or_calibration": False})\n    write_json(output_dir / "mission3_ver4_metrics.json", report)\n    write_json(output_dir / "mission3_ver4_history.json", histories)\n    # 노트북 하나만 Colab에 올려도 제출 코드와 실행 당시 설정을 남긴다.\n    submission = output_dir / "submission"\n    submission.mkdir(exist_ok=True)\n    for name in (*RUNTIME_FILES, "requirements.txt", "README.md"):\n        path = Path(__file__).parent / name\n        if path.exists():\n            shutil.copy2(path, submission / path.name)\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    print("Checkpoint:", checkpoint_path, flush=True)\n    return checkpoint_path, report\n\n\ndef load_manifest(checkpoint_path):\n    manifest = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if manifest.get("format") != "mission3_ver4" or manifest.get("version") != 1:\n        raise ValueError("지원하지 않는 checkpoint입니다. ver4의 manifest .ckpt를 지정하세요.")\n    if manifest["target_symptoms"] != list(TARGET_SYMPTOMS) or not manifest["folds"]:\n        raise ValueError("Checkpoint class order/folds mismatch")\n    root = Path(checkpoint_path).resolve().parent\n    for name in [*manifest["folds"], manifest["sparse"], manifest["tokenizer"]]:\n        path = (root / name).resolve()\n        if not path.is_relative_to(root) or not path.exists():\n            raise FileNotFoundError(f"Checkpoint 구성 파일을 함께 보관해야 합니다: {name}")\n    return manifest\n\n\ndef predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None):\n    """저장된 encoder config/tokenizer만 사용하므로 학습 후 오프라인 추론 가능."""\n    checkpoint_path = Path(checkpoint_path).resolve()\n    manifest = load_manifest(checkpoint_path)\n    root = checkpoint_path.parent\n    config = dict(manifest["config"])\n    config["device"] = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    if batch_size is not None:\n        config["inference_batch_size"] = batch_size\n    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)\n    tokenizer = AutoTokenizer.from_pretrained(root / manifest["tokenizer"], use_fast=True, local_files_only=True)\n    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])\n    neural_probs = np.zeros((len(paths), len(TARGET_SYMPTOMS)), dtype=np.float64)\n    for name in manifest["folds"]:\n        print("Predict:", name, flush=True)\n        neural_probs += predict_checkpoint(root / name, documents, tokenizer, config)\n        _release_memory()\n    neural_probs /= len(manifest["folds"])\n    sparse = joblib.load(root / manifest["sparse"])\n    sparse_probs = predict_sparse(sparse, texts)\n    probabilities = apply_calibrator(neural_probs, sparse_probs, manifest["calibrator"])\n    predictions = write_predictions(paths, probabilities, manifest["calibrator"]["thresholds"], Path(output))\n    print(f"{len(paths):,} predictions saved: {output}", flush=True)\n    return paths, probabilities, predictions\n\n\ndef evaluate_csv(label_dir, prediction_path):\n    """이미 저장된 예측을 파일명으로 정답에 연결한다. 모델/임계값 변경 없음."""\n    with Path(prediction_path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != ["label file name", "symptom"]:\n            raise ValueError("CSV header must be: label file name,symptom")\n        rows = list(reader)\n    by_name = {}\n    for row in rows:\n        if row["label file name"] in by_name:\n            raise ValueError("Duplicate prediction filename")\n        labels = ast.literal_eval(row["symptom"])\n        if not isinstance(labels, list) or any(x not in TARGET_SYMPTOMS for x in labels) or len(set(labels)) != len(labels):\n            raise ValueError("CSV symptom must be a list of unique target strings")\n        by_name[row["label file name"]] = [int(x in labels) for x in TARGET_SYMPTOMS]\n    paths, _, targets = load_dataset(Path(label_dir), include_targets=True)\n    if set(by_name) != {path.name for path in paths}:\n        raise ValueError("CSV와 JSON 파일명 집합이 일치하지 않습니다.")\n    return score_predictions(targets, np.asarray([by_name[path.name] for path in paths], dtype=np.int8))\n', 'ver4_runtime/requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy>=1.26,<3\nscipy>=1.11,<2\nscikit-learn>=1.4,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\n', 'ver4_runtime/train.py': '"""Run all Training-only fitting and calibration; never accepts Validation."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import train_pipeline\nelse:\n    from pipeline import train_pipeline\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label_dir", required=True, type=Path)\n    parser.add_argument("--output_dir", required=True, type=Path)\n    parser.add_argument("--config", type=Path, help="Optional JSON overrides of DEFAULT_CONFIG")\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text(encoding="utf-8")) if args.config else None\n    train_pipeline(args.label_dir, args.output_dir, config)\n\n\nif __name__ == "__main__":\n    main()\n'}
runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver9_runtime_"))
for name, source in RUNTIME_SOURCES.items():
    destination = runtime_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
# 패키지 이름을 고유하게 만들어 이전 ver8 또는 재실행 import와 섞이지 않습니다.
package_name = runtime_dir.name
spec = importlib.util.spec_from_file_location(package_name, runtime_dir / "__init__.py",
                                              submodule_search_locations=[str(runtime_dir)])
module = importlib.util.module_from_spec(spec)
sys.modules[package_name] = module
spec.loader.exec_module(module)
import importlib
supervisor = importlib.import_module(package_name + ".run")
SETTINGS = {
    "input_zip": INPUT_ZIP, "output_dir": OUTPUT_DIR,
    "run_validation": RUN_VALIDATION,
    "run_candidate_diagnostic": RUN_CANDIDATE_DIAGNOSTIC,
    "train_config": TRAIN_CONFIG, "inference_batch_size": INFERENCE_BATCH_SIZE,
}
print("실행 코드 준비 완료:", runtime_dir)


In [ ]:
# 설치부터 마지막 평가까지 단 한 번 호출합니다. 각 단계는 남은 시간만 사용합니다.
result_dir = supervisor.run(SETTINGS, start_epoch=RUN_STARTED)


In [ ]:
# 이미 확정된 결과를 표시하며 이 값으로 모델을 재선택하지 않습니다.
result_dir = Path(OUTPUT_DIR)
for name in ("training_metrics.json", "validation_metrics.json", "candidate_validation_metrics.json", "runtime.json"):
    value = supervisor.LAST_REPORT.get(name)
    if value is None:
        print(name, "— 미생성 (시간 부족/실행 로그 확인)")
        continue
    if "macro_f1" in value:
        print(f"{name}: Macro F1 = {value['macro_f1']:.6f}")
        for label, metrics in value.get("per_symptom", {}).items():
            print(f"  {label}: F1={metrics['f1']:.4f}")
    elif name == "training_metrics.json":
        print("최종 선택:", value["selected_name"])
        print("채택/미채택 사유:", value.get("calibration", {}).get("fallback_reason", value.get("reason")))
        print("학습된 후보:", value.get("candidate_models", []))
    else:
        print(f"전체 경과: {value['elapsed_seconds'] / 60:.1f}분 / 180분")
        print("단계별 분:", {k: round(v / 60, 2) for k, v in value['stages_seconds'].items()})
        print("중단 단계:", value["failures"])
print("제출 실행 코드:", result_dir / "submission")
print("최종 manifest:", result_dir / "mission3_ver9_model.ckpt")
print("최종 CSV:", result_dir / "inference.csv")


## 결과 보관·제출

실행을 마친 노트북을 출력과 함께 저장하세요. `ver9` 결과 폴더 전체에 실제 `.ckpt` 가중치와 tokenizer가 있습니다.
`mission3_ver9_model.ckpt`만 복사하면 추론할 수 없습니다.
제출 실행 코드는 `ver9/submission/`에 있습니다.

```bash
python submission/inference.py --audio_dir /path/audio --label_dir /path/json --ckpt_path mission3_ver9_model.ckpt --output inference.csv
```

CSV 헤더는 `label file name,symptom`입니다. 원래 ver4 폴더의 절대 경로는 필요 없습니다.
상세 모델 설명·논문·문제 조건은 `submission/README.md`, `submission/RESEARCH.md`를 참고하세요.
문제에서 요구하는 실행 로그가 포함된 학습 노트북과 결과 PPT는 실제 결과를 확인한 뒤 함께 준비하세요.
